# Double DQN Platformer Training Notebook

#### Note: Some code results saved here might not reflect the actual code trained, since I did retrain some models hence some of the results might not be the same

This notebook trains a **Double Deep Q-Network (DDQN)** agent to play a custom Godot platformer over a WebSocket bridge. It supports multi-experiment automation, prioritized experience replay, safe interruption/resume, structured training + evaluation statistics, and modular graph generation.

## What this notebook does
1. Connects to a running Godot instance over WebSocket and drives up to 4 players in parallel per episode.
2. Trains a DDQN agent (separate policy/target networks, prioritized replay buffer) against the environment.
3. Logs per-player, per-episode statistics to CSV (training) and separately for greedy evaluation (testing).
4. Generates modular, toggleable graphs for training diagnostics, gameplay behaviour, and cross-model comparison.
5. Supports safe `KeyboardInterrupt` handling — training checkpoints and closes cleanly, and can resume later.

## Notebook structure
| Cell | Purpose |
|---|---|
| 1 | Imports & setup |
| — | Reproducibility / global seeding utility |
| 2 | Default training configuration (grouped) |
| 3 | `DQN` network definition |
| 4 | `SumTree` + prioritized `ReplayBuffer` |
| 5 | `DQNAgent` (Double DQN logic, exploration, checkpointing) |
| 6 | `TrainingStatistics` CSV logger |
| 7 | `PlatformerEnvironment` — Godot WebSocket protocol client |
| 8 | `run_training()` — the asynchronous training loop (interrupt-safe) |
| 9 | `build_config()` / `run_all_experiments()` — multi-experiment automation |
| 10 | Experiment definitions (edit this to configure runs) |
| 11 | **Entry point** — run all configured experiments |
| 12 | Aggregate statistical analysis / report tables |
| — | Modular plotting library (`PLOT_CONFIG` + `plot_*` functions) |
| 13 | Per-model training/gameplay graphs (uses the modular plotting library) |
| 14 | Cross-model comparative overlay graphs |
| 15 | Automated greedy (epsilon=0) evaluation/testing suite |
| 16 | Evaluation result graphs |
| 17 | Overfitting / Underfitting testing for evaluation suite |

**Run instructions:** Execute cells top-to-bottom. Define your experiments in Cell 10 and execute Cell 11 to start training. Cells 12–16 can be re-run independently at any time afterwards purely from the saved CSV logs — they do not require the environment or GPU.

**Stopping training:** Interrupt the kernel (Jupyter's stop/interrupt button, or `KeyboardInterrupt`) at any time during Cell 11. The current episode's checkpoint is saved before the process exits — see Cell 8's documentation for details.


### Cell 1 — Imports & Setup
Standard library, numerical/plotting, PyTorch, and `websockets` imports used throughout the notebook.

- **Inputs:** none.
- **Outputs:** module bindings used by every later cell.
- **Note on `CUBLAS_WORKSPACE_CONFIG`:** this must be set *before* `torch` is imported — it is required by `torch.use_deterministic_algorithms(True)` (enabled later in Cell 8) for deterministic CUDA GEMM operations. Setting it after import has no effect, which is why it appears at the top of this cell rather than next to the seeding code.


In [1]:
import asyncio
import csv
import json
import math
import os
import random
import time
from pathlib import Path

# Must be set before importing torch for deterministic CUDA GEMM behavior
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import websockets
import socket

print("Imports successful. PyTorch CUDA available:", torch.cuda.is_available())

Imports successful. PyTorch CUDA available: True


In [2]:
class Profiler:
    def __init__(self):
        self.times = {
            "godot_wait": 0.0,
            "action_select": 0.0,
            "train_step": 0.0,
            "checkpoint_save": 0.0,
            "other": 0.0,
        }

        self.counts = {
            "steps": 0,
            "updates": 0,
        }

    def add(self, key, dt):
        if key not in self.times:
            self.times[key] = 0.0
        self.times[key] += dt

    def report(self, total_wall):
        print(f"Total: {total_wall:.1f}s")

        for key, value in self.times.items():
            percentage = (
                100 * value / total_wall
                if total_wall > 0
                else 0.0
            )

            print(
                f"  {key}: {value:.1f}s "
                f"({percentage:.1f}%)"
            )

        if self.counts["steps"]:
            print(
                f"  RL steps/sec: "
                f"{self.counts['steps'] / total_wall:.2f}"
            )

        if self.counts["updates"]:
            print(
                f"  updates/sec: "
                f"{self.counts['updates'] / total_wall:.1f}"
            )

prof = Profiler()
# wrap: t0=time.perf_counter(); responses,_ = await env.step(actions); prof.add("godot_wait", time.perf_counter()-t0)
# wrap: t0=time.perf_counter(); loss = agent.train_step(); prof.add("train_step", time.perf_counter()-t0); prof.counts["updates"]+=1

### Reproducibility — Global Seeding Utility
The original notebook already seeded `random`, `torch`, and `torch.cuda` inline at the top of `run_training()`, and enabled deterministic CUDA kernels. This cell factors that identical logic into a single `set_global_seed()` function so it isn't duplicated and is easy to audit — **the behaviour is unchanged**, `run_training()` now just calls this function instead of repeating the same 8 lines.

**Seed taxonomy used in this notebook** (important — these are deliberately separate):
- **Global training seed** (`cfg['seed']`): seeds Python's `random`, `torch`, and `torch.cuda` once per `run_training()` call. This governs network weight initialisation (via `torch`'s default init at construction time) and any global `random` usage.
- **Environment seed**: the same `cfg['seed']` value is also passed to `env.reset(seed)` every episode. This seeds the Godot level's procedural generation so **the same seed always produces the same level layout** — this is a deliberate design choice (train repeatedly on a fixed, reproducible level), not an accidental per-episode RNG reset. Episode-to-episode variation still comes from the agent's evolving policy and its exploration RNG, so this does not make episodes identical to each other.
- **Action-selection / exploration RNG** (`DQNAgent.action_rng`): a *dedicated* `random.Random(seed)` instance per agent, independent of the global `random` module, so exploration behaviour is reproducible even if other code elsewhere calls the global `random` module.
- **Replay-buffer sampling RNG** (`DQNAgent.replay_rng`): a separate `random.Random(seed + 1)` instance, kept apart from the action RNG so that changing one doesn't perturb the other's stream.
- **Evaluation seed**: `run_testing_suite()` resets each test episode with `cfg['seed'] + ep * 100`, giving each of the 10 greedy-evaluation episodes its own distinct, reproducible level layout.

**Determinism vs. performance trade-off:** `torch.use_deterministic_algorithms(True)` together with `cudnn.deterministic = True` and disabling TF32 forces bit-for-bit reproducible GPU kernels, at the cost of disabling some faster non-deterministic CUDA kernels and TensorFloat-32 matmul acceleration — this can meaningfully slow down training on modern GPUs. This notebook keeps that trade-off exactly as it was in the original (favouring reproducibility), since changing it would alter training speed characteristics; if you need faster throughput and can tolerate run-to-run GPU nondeterminism, comment out the four `torch.backends...` / `use_deterministic_algorithms` lines inside `set_global_seed()`.

The active seed is already recorded per experiment (in `cfg['seed']`, in the console banner printed by `run_all_experiments()`, and in the model/statistics file paths), so every CSV/checkpoint can be traced back to its configuration.


In [3]:
def set_global_seed(seed: int) -> None:
    """Seeds every global RNG used for reproducibility and enables deterministic CUDA execution.

    This is identical to the seeding logic previously inlined at the top of run_training() — it has been
    extracted into a function purely for clarity/reuse, with no behavioural change.

    Parameters:
        seed: the global training seed (cfg["seed"]).

    Side effects:
        - Seeds the global `random` module.
        - Seeds PyTorch's CPU and (if available) all CUDA RNGs.
        - Forces deterministic cuDNN/cuBLAS kernels and disables TF32 matmul acceleration.

    Does NOT seed DQNAgent's dedicated action/replay RNGs — those are seeded independently inside
    DQNAgent.__init__ from the same cfg["seed"], so exploration and replay sampling remain reproducible
    without being coupled to (or perturbed by) any other global `random` usage.
    """
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True)

print("set_global_seed() defined.")

set_global_seed() defined.


### Cell 2 — Default Training Configuration
A single flat dict holding every tunable value, grouped by concern with an explanation of what each setting controls and its trade-offs. `build_config()` (Cell 9) later merges per-experiment overrides on top of this.

**Effect of key settings if you change them:**
- `state_size` / `action_size`: must exactly match the Godot side of the WebSocket protocol. Changing these without also changing Godot breaks the protocol and invalidates existing checkpoints (network shape mismatch).
- `hidden_size` / `hidden_layers`: bigger = more model capacity and VRAM/compute per step, but not necessarily better; also makes old checkpoints incompatible (different parameter shapes).
- `learning_rate`: higher trains faster but risks instability/divergence; lower is more stable but slower to converge. Does not affect reproducibility by itself (still deterministic for a fixed seed).
- `replay_buffer_size`: larger buffer uses more RAM (it stores raw transitions) and gives more diverse training batches, but takes longer to "fill" and checkpoints containing the buffer become larger on disk.
- `batch_size`: larger batches use more VRAM per training step and smooth gradient noise; smaller batches are cheaper but noisier.
- `min_replay_size`: how many transitions must be collected before training starts; purely a warm-up gate, does not affect final behaviour once past it.
- `max_episodes` / `max_steps_per_episode`: bound total training length; increasing either simply extends training time roughly linearly.
- `timeout_penalty`: reward applied when an episode is truncated by the step limit — part of the reward design, left untouched.
- `training_updates_per_timestep`: gradient updates performed per environment step; more updates = faster learning per episode but more GPU compute per step (roughly linear slowdown).
- `epsilon_start` / `epsilon_end` / `epsilon_decay`: exploration schedule; unrelated to reproducibility as long as the seed is fixed (the decay is deterministic).
- `target_update_frequency`: how often (in gradient steps) the target network is synced from the policy network — a core DDQN hyperparameter, left untouched.
- `seed`: the global reproducibility seed (see the Reproducibility cell above). Changing it produces a different — but for that seed, repeatable — training run.
- `save_frequency`: how often a "Model saved." status line is printed (a checkpoint is actually written every episode regardless, so training can always be resumed at the last completed episode).
- `resume_training`: if `True` and a checkpoint already exists at `model_path`, training resumes from it instead of starting fresh.
- `model_path` / `statistics_path`: output locations; auto-derived per experiment name unless explicitly overridden (see `build_config()`).
- `console_log_frequency`: **new, purely cosmetic** — controls how often the per-episode console summary is printed (every Nth episode). Default `25` prints a compact progress-bar summary every 25th episode instead of flooding the console every single episode; set it to `1` to restore the original print-every-episode behaviour. Every episode is still fully logged to CSV and checkpointed regardless of this setting.


In [20]:
DEFAULT_CONFIG = {
    # Environment & Protocol
    "state_size": 16,
    "action_size": 6,
    "player_ids": [0, 1, 2, 3],
    "websocket_uri": "ws://localhost:8765",
    "websocket_timeout": 10.0,

    # Network Architecture
    "hidden_size": 128,
    "hidden_layers": 2,

    # Optimization & Learning
    "learning_rate": 0.0003,
    "gamma": 0.99,

    # Replay Buffer
    "replay_buffer_size": 300_000,
    "batch_size": 64,
    "min_replay_size": 2_000,

    # Training & Step Penalties
    "max_episodes": 1_000,
    "max_steps_per_episode": 400,
    "timeout_penalty": -5.0,
    "training_updates_per_timestep": 4,

    # Exploration
    "epsilon_start": 1.0,
    "epsilon_end": 0.05,
    "epsilon_decay": 0.995,

    # Target Network
    "target_update_frequency": 2_000,

    # Reproducibility
    "seed": 42,

    # Logging
    "console_log_frequency": 25,  # print a progress-bar summary every Nth episode instead of flooding the console every episode

    # Persistence
    "save_frequency": 100,
    "resume_training": True,
    "model_path": "results/models/ddqn_optimised.pth",
    "statistics_path": "results/data/training_results_optimised.csv",
}

print(f"Loaded DEFAULT_CONFIG with {len(DEFAULT_CONFIG)} configuration options.")

Loaded DEFAULT_CONFIG with 26 configuration options.


### Cell 3 — Neural Network Definition (DQN)
A plain feed-forward MLP: `state_size -> hidden_size -> ... -> action_size`, with `hidden_layers` fully-connected + ReLU blocks. This same architecture is instantiated twice by `DQNAgent` (policy net and target net) — that pairing is what makes this a **Double** DQN, not a property of the network class itself.

- **Inputs:** a state vector of length `state_size` (batched).
- **Outputs:** one Q-value per action (`action_size` outputs).
- **Side effects:** none — a pure `nn.Module`.


In [5]:
class DQN(nn.Module):
    """Deep Q-Network parameterized by active experiment configuration."""
    def __init__(self, cfg):
        super().__init__()
        state_size = cfg["state_size"]
        action_size = cfg["action_size"]
        hidden_size = cfg["hidden_size"]
        hidden_layers = cfg["hidden_layers"]

        layers = [nn.Linear(state_size, hidden_size), nn.ReLU()]
        for _ in range(hidden_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(hidden_size, action_size))
        self.network = nn.Sequential(*layers)

    def forward(self, state):
        return self.network(state)

print("DQN network architecture initialized.")

DQN network architecture initialized.


### Cell 4 — Prioritized Replay Buffer & SumTree
`SumTree` is a binary-tree structure supporting O(log n) prioritized sampling and priority updates. `ReplayBuffer` wraps it to implement **Prioritized Experience Replay (PER)**: transitions with larger TD error are sampled more often, with importance-sampling weights (`beta`, annealed from `beta_start` to 1.0 over `beta_frames`) correcting for the resulting sampling bias.

- **`add(...)`:** stores a new transition at the current max priority, so it's guaranteed to be sampled at least once before its real TD-error priority is known.
- **`sample(batch_size)`:** stratified sampling — the total priority mass is split into `batch_size` equal segments and one sample is drawn per segment, which spreads samples across the whole priority distribution rather than clustering on the highest-priority items. Returns `(batch, idxs, weights)`.
- **`update_priorities(idxs, td_errors)`:** called after each training step to write back the new TD-error-derived priorities.
- **`rng`:** the buffer takes an externally supplied RNG (`DQNAgent.replay_rng`) rather than using the global `random` module — this is what keeps replay sampling reproducible and decoupled from the exploration RNG (see the Reproducibility cell).
- **`state_dict()` / `load_state_dict()`:** full serialization of the tree and its metadata so training can resume with the *exact* same buffer contents and sampling state, not just the network weights.

**Performance note:** `replay_buffer_size` transitions are kept as plain Python lists/tuples rather than a contiguous NumPy/Tensor array. This is simple and correct but means `sample()` does per-item Python-level work; it has been left unchanged since converting it to a vectorised NumPy-backed ring buffer would change floating-point accumulation order and priority tie-breaking behaviour (a subtle output-changing risk), which the brief for this pass explicitly rules out.


In [6]:
class SumTree:
    def __init__(self, capacity):
        self.capacity = capacity
        self.tree = [0.0] * (2 * capacity - 1)
        self.data = [None] * capacity
        self.write = 0
        self.n_entries = 0

    def _propagate(self, idx, change):
        parent = (idx - 1) // 2
        self.tree[parent] += change
        if parent != 0:
            self._propagate(parent, change)

    def _retrieve(self, idx, s):
        left = 2 * idx + 1
        right = left + 1
        if left >= len(self.tree):
            return idx
        if s <= self.tree[left]:
            return self._retrieve(left, s)
        return self._retrieve(right, s - self.tree[left])

    def total(self):
        return self.tree[0]

    def add(self, p, data):
        idx = self.write + self.capacity - 1
        self.data[self.write] = data
        self.update(idx, p)
        self.write = (self.write + 1) % self.capacity
        if self.n_entries < self.capacity:
            self.n_entries += 1

    def update(self, idx, p):
        change = p - self.tree[idx]
        self.tree[idx] = p
        self._propagate(idx, change)

    def get(self, s):
        idx = self._retrieve(0, s)
        data_idx = idx - self.capacity + 1
        return idx, self.tree[idx], self.data[data_idx]


class ReplayBuffer:
    """Prioritized Experience Replay buffer."""
    def __init__(self, capacity, rng, alpha=0.6, beta_start=0.4, beta_frames=100_000):
        self.capacity = capacity
        self.rng = rng
        self.alpha = alpha
        self.beta = beta_start
        self.beta_start = beta_start
        self.beta_frames = beta_frames
        self.frame_count = 0
        self.tree = SumTree(capacity)
        self.epsilon = 1e-5
        self.max_priority = 1.0

    def add(self, state, action, reward, next_state, done):
        self.tree.add(self.max_priority, (state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch, idxs, priorities = [], [], []
        segment = self.tree.total() / batch_size
        self.frame_count += 1
        self.beta = min(1.0, self.beta_start + self.frame_count * (1.0 - self.beta_start) / self.beta_frames)

        for i in range(batch_size):
            s = self.rng.uniform(segment * i, segment * (i + 1))
            idx, p, data = self.tree.get(s)
            if data is None:
                data = next(d for d in self.tree.data if d is not None)
            batch.append(data)
            idxs.append(idx)
            priorities.append(p)

        total_p = self.tree.total()
        n = self.tree.n_entries
        weights = [
            (1.0 / (n * (p / total_p))) ** self.beta if p > 0 else 0.0
            for p in priorities
        ]
        max_weight = max(weights) if weights else 1.0
        weights = [w / max_weight for w in weights] if max_weight > 0 else [1.0] * batch_size
        return batch, idxs, weights

    def update_priorities(self, idxs, td_errors):
        for idx, td_error in zip(idxs, td_errors):
            p = (abs(td_error) + self.epsilon) ** self.alpha
            self.tree.update(idx, p)
            self.max_priority = max(self.max_priority, p)

    def __len__(self):
        return self.tree.n_entries

    def state_dict(self):
        return {
            "capacity": self.capacity,
            "tree_tree": self.tree.tree,
            "tree_data": self.tree.data,
            "tree_write": self.tree.write,
            "tree_n_entries": self.tree.n_entries,
            "max_priority": self.max_priority,
            "frame_count": self.frame_count,
            "beta": self.beta,
        }

    def load_state_dict(self, state):
        self.capacity = int(state.get("capacity", self.capacity))
        self.tree = SumTree(self.capacity)
        if "tree_tree" in state:
            self.tree.tree = list(state["tree_tree"])
            self.tree.data = list(state["tree_data"])
            self.tree.write = int(state["tree_write"])
            self.tree.n_entries = int(state["tree_n_entries"])
            self.max_priority = float(state.get("max_priority", self.max_priority))
            self.frame_count = int(state.get("frame_count", self.frame_count))
            self.beta = float(state.get("beta", self.beta))
        else:
            for exp in state.get("experiences", []):
                self.add(exp[0], exp[1], exp[2], exp[3], exp[4])

print("ReplayBuffer & SumTree defined.")

ReplayBuffer & SumTree defined.


### Cell 5 — Double DQN Agent
`DQNAgent` owns the policy/target networks, optimizer, replay buffer, exploration state, and per-player adaptive exploration/performance tracking (up to 4 simultaneous players).

**Key methods:**
- `choose_actions(states_by_player, epsilon=None)`: batched forward pass through `policy_net` for all active players at once (single GPU call instead of one per player — see performance notes below), then per-player epsilon-greedy selection. Two extra behaviours layered on top of plain epsilon-greedy:
  - **Timed exploration commitment:** once an exploratory action is chosen, it is *held* for a random TTL (`exploration_min_ttl`–`exploration_max_ttl` steps) via `_roll_exploration_action`, rather than re-rolling a fresh random action every single step. This produces more meaningful directed exploration (e.g. actually walking somewhere) instead of jittering in place.
  - **Per-player adaptive epsilon:** a player whose reward moving average (`player_rewards_ma`) is lagging the group average gets a temporarily boosted epsilon (up to +50%, floor -20%), and any player in a particular state (`state[15] > 0.5`, an environment-reported flag) gets a forced minimum epsilon of 0.15. This is existing reward-shaping-adjacent behaviour and has been left untouched.
- `train_step()`: standard Double DQN update — actions are *selected* by the policy network and *evaluated* by the target network (this decoupling is what distinguishes DDQN from vanilla DQN and reduces overestimation bias), using Smooth-L1 (Huber) loss weighted by PER importance-sampling weights. Also feeds TD errors back into the replay buffer's priorities, and periodically hard-syncs the target network.
- `save(path, completed_episode)` / `load(path)`: atomic checkpoint write (via a `.tmp` file + `os.replace`, so a crash mid-save can never corrupt the previous checkpoint) capturing everything needed to resume training bit-for-bit: both networks, optimizer state, replay buffer contents, epsilon, training step count, per-player performance trackers, and **all RNG states** (`action_rng`, `replay_rng`, global `random`, CPU `torch`, and CUDA if available). `load()` returns `None` (triggering a fresh start) on a network shape mismatch instead of silently loading incompatible weights.

- **`action_rng` / `replay_rng`:** dedicated `random.Random` instances seeded from `cfg["seed"]` and `cfg["seed"] + 1` respectively — kept separate from each other and from the global `random` module (see the Reproducibility cell for why).


In [7]:
JUMP_ACTIONS = {4, 5}
EXPLORATION_ACTION_WEIGHTS = [1, 1, 1, 1, 1, 1]


class DQNAgent:
    """Double DQN Agent with dynamic player exploration adaptivity."""
    def __init__(self, cfg, device):
        self.cfg = cfg
        self.device = device
        seed = cfg["seed"]

        self.action_rng = random.Random(seed)
        self.replay_rng = random.Random(seed + 1)

        self.policy_net = DQN(cfg).to(device)
        self.target_net = DQN(cfg).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        # --- FASTER: fused Adam collapses the per-parameter-tensor update loop (one
        # kernel launch per weight/bias tensor) into one or two fused kernels. Identical
        # Adam math, just fewer/larger kernel launches -- meaningfully cuts optimizer.step()
        # overhead at small batch sizes where launch overhead dominates actual compute.
        # Falls back cleanly on CPU-only runs or older PyTorch builds without fused Adam.
        try:
            self.optimizer = optim.Adam(
                self.policy_net.parameters(), lr=cfg["learning_rate"],
                fused=(device.type == "cuda")
            )
        except TypeError:
            self.optimizer = optim.Adam(self.policy_net.parameters(), lr=cfg["learning_rate"])
        # --- END ---
        self.replay_buffer = ReplayBuffer(cfg["replay_buffer_size"], self.replay_rng)

        self.epsilon = cfg["epsilon_start"]
        self.exploration_min_ttl = 3
        self.exploration_max_ttl = 8
        self._exploration_action = {}
        self._exploration_kind = {}
        self._exploration_ttl = {}

        self.training_steps = 0
        self.last_selection_by_player = {}
        self.last_train_metrics = {}
        self.player_rewards_ma = {0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0}
        self.player_episodes_count = {0: 0, 1: 0, 2: 0, 3: 0}

    def _roll_exploration_action(self, player_id):
        action = self.action_rng.choices(
            range(self.cfg["action_size"]),
            weights=EXPLORATION_ACTION_WEIGHTS,
            k=1
        )[0]
        self._exploration_action[player_id] = action
        self._exploration_kind[player_id] = "timed"
        self._exploration_ttl[player_id] = self.action_rng.randint(
            self.exploration_min_ttl, self.exploration_max_ttl
        ) - 1
        return action

    def choose_actions(self, states_by_player, epsilon=None):
        player_ids = list(states_by_player)
        actions = {}
        effective_epsilon = self.epsilon if epsilon is None else float(epsilon)
        if not player_ids:
            self.last_selection_by_player = {}
            return actions

        state_tensor = torch.tensor(
            [states_by_player[pid] for pid in player_ids],
            dtype=torch.float32, device=self.device
        )
        with torch.no_grad():
            q_values = self.policy_net(state_tensor).detach().cpu()
        greedy_actions = q_values.argmax(dim=1).tolist()

        self.last_selection_by_player = {}
        for row, player_id in enumerate(player_ids):
            player_epsilon = effective_epsilon
            if all(c > 2 for c in self.player_episodes_count.values()):
                avg_reward = sum(self.player_rewards_ma.values()) / 4.0
                player_perf = self.player_rewards_ma.get(player_id, 0.0)
                if abs(avg_reward) > 1e-5:
                    scale = 1.0 + max(-0.2, min(0.5, (avg_reward - player_perf) / abs(avg_reward)))
                    player_epsilon = max(self.cfg["epsilon_end"], min(1.0, effective_epsilon * scale))

            if states_by_player[player_id][15] > 0.5:
                player_epsilon = max(player_epsilon, 0.15)

            kind = self._exploration_kind.get(player_id)
            ttl = self._exploration_ttl.get(player_id, 0)
            exploratory = False
            action = None

            if kind == "timed":
                if ttl > 0:
                    exploratory = True
                    action = self._exploration_action[player_id]
                    self._exploration_ttl[player_id] = ttl - 1
                else:
                    kind = None

            if kind is None:
                exploratory = self.action_rng.random() < player_epsilon
                if exploratory:
                    action = self._roll_exploration_action(player_id)
                else:
                    action = greedy_actions[row]
                    self._exploration_kind[player_id] = None
                    self._exploration_ttl[player_id] = 0

            values = q_values[row]
            actions[player_id] = action
            self.last_selection_by_player[player_id] = {
                "exploratory": exploratory,
                "q_sum": float(values.sum()),
                "q_squared_sum": float(values.square().sum()),
                "q_min": float(values.min()),
                "q_max": float(values.max()),
                "q_count": int(values.numel()),
            }
        return actions

    def remember(self, state, action, reward, next_state, done):
        self.replay_buffer.add(state, action, reward, next_state, done)

    def train_step(self):
        cfg = self.cfg
        if len(self.replay_buffer) < cfg["min_replay_size"]:
            return None

        batch, idxs, weights = self.replay_buffer.sample(cfg["batch_size"])
        states, actions, rewards, next_states, dones = zip(*batch)

        # --- FASTER: go through NumPy once instead of nested-Python-list iteration ---
        states = torch.from_numpy(np.asarray(states, dtype=np.float32)).to(self.device)
        next_states = torch.from_numpy(np.asarray(next_states, dtype=np.float32)).to(self.device)
        actions = torch.from_numpy(np.asarray(actions, dtype=np.int64)).to(self.device)
        rewards = torch.from_numpy(np.asarray(rewards, dtype=np.float32)).to(self.device)
        dones = torch.from_numpy(np.asarray(dones, dtype=np.float32)).to(self.device)
        weights_tensor = torch.from_numpy(np.asarray(weights, dtype=np.float32)).to(self.device)
        # --- END ---

        current_q_values = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            next_actions = self.policy_net(next_states).argmax(dim=1)
            next_q_values = self.target_net(next_states).gather(1, next_actions.unsqueeze(1)).squeeze(1)
            target_q_values = rewards + cfg["gamma"] * next_q_values * (1 - dones)

        td_errors_tensor = (target_q_values - current_q_values).detach()

        loss_per_element = F.smooth_l1_loss(current_q_values, target_q_values, reduction="none")
        loss = (loss_per_element * weights_tensor).mean()

        self.optimizer.zero_grad(set_to_none=True)  # skip the memset kernel per param tensor
        loss.backward()
        self.optimizer.step()

        # --- FASTER: one combined device->host transfer instead of three separate .item()/.cpu() calls ---
        mean_abs_td_error_tensor = td_errors_tensor.abs().mean().unsqueeze(0)
        combined = torch.cat([td_errors_tensor, mean_abs_td_error_tensor, loss.detach().unsqueeze(0)]).cpu().numpy()
        td_errors = combined[:-2]
        mean_abs_td_error = float(combined[-2])
        loss_value = float(combined[-1])
        # --- END ---

        self.replay_buffer.update_priorities(idxs, td_errors)

        self.training_steps += 1
        if self.training_steps % cfg["target_update_frequency"] == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        self.last_train_metrics = {"loss": loss_value, "mean_abs_td_error": mean_abs_td_error}
        return loss_value

    def decay_epsilon(self):
        self.epsilon = max(self.cfg["epsilon_end"], self.epsilon * self.cfg["epsilon_decay"])

    def reset_exploration_commitments(self):
        self._exploration_ttl.clear()
        self._exploration_action.clear()
        self._exploration_kind.clear()

    def update_player_performance(self, player_id, reward):
        count = self.player_episodes_count.get(player_id, 0)
        self.player_episodes_count[player_id] = count + 1
        alpha = min(0.05, 1.0 / (count + 1))
        self.player_rewards_ma[player_id] = (
            (1.0 - alpha) * self.player_rewards_ma.get(player_id, 0.0) + alpha * reward
        )

    def save(self, path, completed_episode):
        checkpoint = {
            "completed_episode": completed_episode,
            "policy_net": self.policy_net.state_dict(),
            "target_net": self.target_net.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "epsilon": self.epsilon,
            "training_steps": self.training_steps,
            "replay_buffer": self.replay_buffer.state_dict(),
            "action_rng_state": self.action_rng.getstate(),
            "replay_rng_state": self.replay_rng.getstate(),
            "python_rng_state": random.getstate(),
            "torch_rng_state": torch.get_rng_state(),
            "player_rewards_ma": self.player_rewards_ma,
            "player_episodes_count": self.player_episodes_count,
        }
        if torch.cuda.is_available():
            checkpoint["cuda_rng_states"] = torch.cuda.get_rng_state_all()
        tmp = path + ".tmp"
        torch.save(checkpoint, tmp)
        os.replace(tmp, path)

    def load(self, path):
        checkpoint = torch.load(path, map_location=self.device, weights_only=False)
        try:
            self.policy_net.load_state_dict(checkpoint["policy_net"])
            self.target_net.load_state_dict(checkpoint["target_net"])
            self.optimizer.load_state_dict(checkpoint["optimizer"])
        except RuntimeError as e:
            print("Checkpoint shape mismatch; starting fresh training.", e)
            return None

        self.epsilon = float(checkpoint["epsilon"])
        self.training_steps = int(checkpoint.get("training_steps", 0))
        if "player_rewards_ma" in checkpoint:
            self.player_rewards_ma = checkpoint["player_rewards_ma"]
        if "player_episodes_count" in checkpoint:
            self.player_episodes_count = checkpoint["player_episodes_count"]
        if "replay_buffer" in checkpoint:
            self.replay_buffer.load_state_dict(checkpoint["replay_buffer"])
        if "action_rng_state" in checkpoint:
            self.action_rng.setstate(checkpoint["action_rng_state"])
        if "replay_rng_state" in checkpoint:
            self.replay_rng.setstate(checkpoint["replay_rng_state"])
        if "python_rng_state" in checkpoint:
            random.setstate(checkpoint["python_rng_state"])
        if "torch_rng_state" in checkpoint:
            try:
                state = checkpoint["torch_rng_state"]
                if isinstance(state, torch.Tensor):
                    state = state.detach().to(device="cpu", dtype=torch.uint8)
                torch.set_rng_state(torch.as_tensor(state, dtype=torch.uint8, device="cpu"))
            except (TypeError, ValueError, RuntimeError):
                print("CPU RNG state incompatible; using configured seed.")
        if torch.cuda.is_available() and "cuda_rng_states" in checkpoint:
            try:
                cuda_states = [
                    (s.detach().to(device="cpu", dtype=torch.uint8)
                     if isinstance(s, torch.Tensor)
                     else torch.as_tensor(s, dtype=torch.uint8, device="cpu"))
                    for s in checkpoint["cuda_rng_states"]
                ]
                torch.cuda.set_rng_state_all(cuda_states)
            except (TypeError, ValueError, RuntimeError):
                print("CUDA RNG state incompatible; using configured seed.")

        if "completed_episode" not in checkpoint:
            return None
        return int(checkpoint["completed_episode"])

print("DQNAgent class defined.")

DQNAgent class defined.


### Cell 6 — Training Statistics Logger
`TrainingStatistics` is a thin, flush-on-write CSV logger — one row per `(episode, player_id)`. Keeping it this granular (rather than pre-aggregating) is what lets Cells 12–14 recompute any aggregation (mean/rolling average/std, per-player breakdowns, etc.) later without re-running training.

- **`HEADER`:** the fixed CSV schema. Includes reward, score, loss statistics, epsilon, timing, gameplay stats (apples/snails/distance/completion/death cause), exploration-vs-greedy action counts, Q-value statistics, and a full per-action selection histogram — this is the raw material every plot in Cells 13/14 is built from.
- **Schema migration:** if `append=True` and an existing CSV has an older/different header, `_migrate_schema()` backs up the original file (`*.pre_schema_migration.csv`) and rewrites it against the current `HEADER`, filling any newly-added columns with empty values. This means resuming an older experiment after this notebook gains new statistics fields won't crash or silently drop data.
- **`flush()` after every row:** guarantees no data is lost if training is interrupted mid-episode; the trade-off is a small amount of I/O overhead per episode, which is negligible next to the network/training cost per episode and was left as-is.


In [8]:
class TrainingStatistics:
    ACTION_NAMES = ("idle", "left", "right", "jump", "left_jump", "right_jump")
    HEADER = (
        "episode", "player_id", "reward", "score", "loss", "loss_std", "loss_max",
        "td_error", "epsilon", "training_seconds", "episode_length", "distance_travelled",
        "apples_collected", "apple_collection_rate", "snails_avoided", "completed",
        "death_cause", "exploratory_actions", "greedy_actions", "q_mean", "q_max",
        "q_min", "q_std", "action_idle", "action_left", "action_right", "action_jump",
        "action_left_jump", "action_right_jump",
    )

    def __init__(self, path, append=False):
        self.path = path
        directory = os.path.dirname(path)
        if directory:
            os.makedirs(directory, exist_ok=True)
        append_to_existing = append and os.path.exists(path)
        if append_to_existing:
            with open(path, "r", newline="", encoding="utf-8") as f:
                existing_header = tuple(next(csv.reader(f), []))
            if existing_header != self.HEADER:
                self._migrate_schema(path, existing_header)
        self.file = open(path, "a" if append_to_existing else "w", newline="", encoding="utf-8")
        self.writer = csv.writer(self.file)
        if not append_to_existing:
            self.writer.writerow(self.HEADER)
        self.file.flush()

    def add_episode(self, episode, player_id, reward, score, loss, epsilon,
                    episode_length, distance_travelled, apples_collected,
                    apple_collection_rate, snails_avoided, completed, death_cause,
                    loss_std=0.0, loss_max=0.0, td_error=0.0, training_seconds=0.0,
                    exploratory_actions=0, greedy_actions=0,
                    q_mean=0.0, q_max=0.0, q_min=0.0, q_std=0.0, action_counts=None):
        action_counts = list(action_counts or [0] * len(self.ACTION_NAMES))
        if len(action_counts) != len(self.ACTION_NAMES):
            raise ValueError("Expected six action counts")
        self.writer.writerow([
            episode, player_id, reward, score,
            loss, loss_std, loss_max, td_error, epsilon, training_seconds,
            episode_length, distance_travelled,
            apples_collected, apple_collection_rate,
            snails_avoided, completed, death_cause,
            exploratory_actions, greedy_actions,
            q_mean, q_max, q_min, q_std,
            *action_counts,
        ])
        self.file.flush()

    def close(self):
        if hasattr(self, "file") and not self.file.closed:
            self.file.close()

    @classmethod
    def _migrate_schema(cls, path, existing_header):
        backup_path = path + ".pre_schema_migration.csv"
        if not os.path.exists(backup_path):
            with open(path, "r", newline="", encoding="utf-8") as src, \
                 open(backup_path, "w", newline="", encoding="utf-8") as bak:
                bak.write(src.read())
        tmp = path + ".schema_migration.tmp"
        with open(path, "r", newline="", encoding="utf-8") as src, \
             open(tmp, "w", newline="", encoding="utf-8") as dst:
            reader = csv.DictReader(src)
            writer = csv.DictWriter(dst, fieldnames=cls.HEADER)
            writer.writeheader()
            for row in reader:
                writer.writerow({col: row.get(col, "") for col in cls.HEADER})
        os.replace(tmp, path)
        added = sorted(set(cls.HEADER) - set(existing_header))
        print(f"Migrated CSV header. Added: {', '.join(added)}")

print("TrainingStatistics logger defined.")

TrainingStatistics logger defined.


### Cell 7 — WebSocket Environment Interface
`PlatformerEnvironment` is the client half of the Godot WebSocket protocol. Its message formats (`reset`/`reset_response`, `step`/`step_response`, `quit`), player-ID scheme, action-ID scheme, and state encoding are the environment's contract and have **not** been changed.

- **`reset(seed)`:** sends `{type: reset, seed}`, expects `reset_response` with a `states` dict keyed by player ID; validates every returned state before returning it.
- **`step(actions)`:** sends one `step` message per active player *concurrently* (`asyncio.gather`) rather than sequentially — this is an existing performance choice (avoids paying Godot's per-message round-trip latency once per player, per step) and has been preserved.
- **`_validate_state(...)`:** enforces the state contract (a list of exactly `state_size` numeric values, each within `[-1, 1]` with a small floating-point tolerance) and raises immediately on violation, rather than silently accepting a malformed state — this surfaces a broken Godot/protocol mismatch loudly instead of letting corrupted data quietly poison training.
- **Error handling:** connection loss, timeouts, malformed JSON, an explicit `{type: error}` message, mismatched/duplicate/missing player responses, and invalid states all raise a clear `RuntimeError` rather than being swallowed.
- **`close_godot()`:** sends an explicit `{type: quit}` message so the Godot process terminates cleanly instead of being left as an orphaned process when the notebook finishes.


In [9]:
class PlatformerEnvironment:
    """WebSocket client controlling Godot players synchronously."""
    def __init__(self, cfg):
        self.uri = cfg["websocket_uri"]
        self.timeout = cfg["websocket_timeout"]
        self.state_size = cfg["state_size"]
        self.websocket = None

    async def connect(self):
        self.websocket = await asyncio.wait_for(websockets.connect(self.uri), self.timeout)

        try:
            sock = self.websocket.transport.get_extra_info("socket")
            if sock is not None:
                sock.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
        except AttributeError:
            # Older/newer websockets versions expose the raw connection differently;
            # fall back to the underlying protocol's transport.
            sock = self.websocket.protocol.transport.get_extra_info("socket")
            if sock is not None:
                sock.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
        print("Connected to Godot environment!")

    async def reset(self, seed):
        await self._send({"type": "reset", "seed": int(seed)})
        response = await self._receive()
        if response.get("type") != "reset_response" or not isinstance(response.get("states"), dict):
            raise RuntimeError("Godot returned an invalid reset response")
        states = {int(pid): state for pid, state in response["states"].items()}
        for pid, state in states.items():
            self._validate_state(pid, state)
        return states

    async def step(self, actions):
        if not actions:
            return {}, True
        await asyncio.gather(*(
            self._send({"type": "step", "player_id": int(pid), "action": int(action)})
            for pid, action in actions.items()
        ))
        expected = set(actions)
        responses = {}
        for _ in range(len(expected)):
            response = await self._receive()
            if response.get("type") != "step_response":
                raise RuntimeError("Godot returned an invalid step response")
            pid = response.get("player_id")
            if not isinstance(pid, int) or pid not in expected:
                raise RuntimeError(f"Godot returned response for unknown player {pid!r}")
            if pid in responses:
                raise RuntimeError(f"Godot returned duplicate response for player {pid}")
            self._validate_state(pid, response.get("state"))
            if not isinstance(response.get("reward"), (int, float)) or not isinstance(response.get("done"), bool):
                raise RuntimeError(f"Godot returned malformed transition for player {pid}")
            responses[pid] = response
        missing = expected - responses.keys()
        if missing:
            raise RuntimeError(f"Godot did not return responses for players {sorted(missing)}")
        return responses, all(r["done"] for r in responses.values())

    async def _send(self, payload):
        if self.websocket is None:
            raise RuntimeError("Environment is not connected")
        try:
            await asyncio.wait_for(self.websocket.send(json.dumps(payload)), self.timeout)
        except (asyncio.TimeoutError, websockets.ConnectionClosed) as e:
            raise RuntimeError("WebSocket closed or timed out while sending") from e

    async def _receive(self):
        if self.websocket is None:
            raise RuntimeError("Environment is not connected")
        try:
            raw = await asyncio.wait_for(self.websocket.recv(), self.timeout)
            response = json.loads(raw)
        except asyncio.TimeoutError as e:
            raise RuntimeError(f"Timed out after {self.timeout}s waiting for Godot") from e
        except websockets.ConnectionClosed as e:
            raise RuntimeError("Godot closed the WebSocket connection") from e
        except json.JSONDecodeError as e:
            raise RuntimeError("Godot sent malformed JSON") from e
        if not isinstance(response, dict):
            raise RuntimeError("Godot response must be a JSON object")
        if response.get("type") == "error":
            raise RuntimeError("Godot error: " + str(response.get("message", "unknown error")))
        return response

    def _validate_state(self, player_id, state):
        if not isinstance(state, list):
            raise RuntimeError(f"Player {player_id} returned a non-list state")
        if len(state) != self.state_size:
            raise RuntimeError(f"Player {player_id} state size {len(state)}, expected {self.state_size}")
        for i, v in enumerate(state):
            if not isinstance(v, (int, float)):
                raise RuntimeError(f"Player {player_id} state[{i}] is not numeric: {v!r}")
            if not -1.0001 <= v <= 1.0001:
                raise RuntimeError(f"Player {player_id} state[{i}]={v} outside [-1, 1]")

    async def close(self):
        if self.websocket is not None:
            try:
                await self.websocket.close()
            except Exception:
                pass
            self.websocket = None

    async def close_godot(self):
        """Sends a quit signal to Godot so the engine process terminates cleanly."""
        if self.websocket is not None:
            try:
                await self._send({"type": "quit"})
                print("Sent termination signal to Godot process ('OWNAssignment').")
            except Exception:
                pass
            await self.close()

print("PlatformerEnvironment client defined with close_godot().")

PlatformerEnvironment client defined with close_godot().


### Cell 8 — Asynchronous Training Execution Loop
`run_training(cfg, device)` runs one full experiment end-to-end: seeding, optional resume, the episode loop, per-episode logging/checkpointing, and safe interrupt handling.

- **Inputs:** a fully-built `cfg` dict (see `build_config`, Cell 9) and a `torch.device`.
- **Outputs:** a `summary` dict (`name`, `seed`, `final_reward`, `best_reward`, `total_episodes`), and as a side effect a full CSV of per-episode statistics plus a resumable checkpoint on disk.
- **Modifies:** creates a fresh `agent.replay_buffer` / networks (or restores them from checkpoint), writes `cfg['model_path']` every episode, appends every episode to `cfg['statistics_path']`.

**Episode flow:** reset environment for all active players -> loop timesteps -> batched action selection -> step all active players concurrently -> apply timeout penalty on step-limit truncation -> store transition -> `training_updates_per_timestep` gradient updates -> decay epsilon -> log statistics -> checkpoint -> reset for the next episode (pre-fetched at the end of the current one, so there's no idle round-trip at the start of the next episode).

**Safe interruption (`KeyboardInterrupt` / `asyncio.CancelledError`):** caught explicitly around the episode loop. On interrupt, the current episode's progress is checkpointed (`agent.save`, which includes the replay buffer and all RNG states — see Cell 5) and a clear `[INTERRUPT] ... stopped at episode N` message is printed before the exception is re-raised. The `finally` block closes the statistics file handle and the WebSocket connection regardless of how the loop exits (interrupt, exception, or normal completion), so a stopped run never leaves a dangling connection or an unflushed CSV. Because a checkpoint is saved *every* episode during the loop (not just on interrupt), you can also just stop the kernel abruptly and resume from the last completed episode.

**Console output:** the per-player and per-episode summary lines are gated by `cfg['console_log_frequency']` (default `25`) — see Cell 2. Instead of a full episode/reward dump on every printed line, the episode summary now shows `_format_progress_bar()`'s compact `[####----] 62.0% (620/1000)` bar so it's easy to see run progress at a glance during long training runs. This is purely cosmetic: every episode is still fully written to the statistics CSV and checkpointed regardless of this setting, and set `console_log_frequency` to `1` to restore a print on every episode.

**Note on environment reset seeding:** `env.reset(seed)` is called with the *same* `cfg['seed']` at the start of every episode (both at loop entry and when pre-fetching the next episode's start states). This is intentional — it fixes the Godot level layout across the whole run rather than randomising it every episode — see the Reproducibility cell for the full explanation of why this doesn't make episodes identical.


In [10]:
def _format_progress_bar(episode, max_episodes, width=24):
    """Renders a simple text progress bar such as "[###########-------------]  47.0% (470/1000)".

    Purely cosmetic -- used only for the throttled console summary; has no effect on training.
    """
    fraction = min(1.0, episode / max_episodes) if max_episodes else 1.0
    filled = int(round(fraction * width))
    bar = "#" * filled + "-" * (width - filled)
    return f"[{bar}] {fraction * 100:5.1f}% ({episode}/{max_episodes})"


async def run_training(cfg, device):
    """Runs one full training experiment using cfg with clean interrupt handling."""
    seed = cfg["seed"]
    set_global_seed(seed)  # seeds random/torch/cuda + deterministic kernels (see Reproducibility cell)

    env = PlatformerEnvironment(cfg)
    agent = DQNAgent(cfg, device)

    prof_times = {"godot_wait": 0.0, "train_step": 0.0, "checkpoint_save": 0.0}
    prof_counts = {"steps": 0, "updates": 0}

    completed_episode = 0
    resume = cfg["resume_training"] and os.path.isfile(cfg["model_path"])
    if resume:
        restored = agent.load(cfg["model_path"])
        if restored is None:
            resume = False
            print("Loaded weights, but no resumable episode checkpoint found.")
        else:
            completed_episode = restored
            print(f"Resumed checkpoint after episode {completed_episode}.")

    statistics = TrainingStatistics(cfg["statistics_path"], append=resume)
    await env.connect()

    summary = {
        "name": cfg.get("name", "experiment"),
        "seed": seed,
        "final_reward": None,
        "best_reward": float("-inf"),
        "total_episodes": 0,
    }

    current_ep = completed_episode
    try:
        next_episode_states = None
        for episode in range(completed_episode + 1, cfg["max_episodes"] + 1):
            current_ep = episode
            episode_started_at = time.perf_counter()
            states = (
                next_episode_states
                if next_episode_states is not None
                else await env.reset(seed)
            )
            agent.reset_exploration_commitments()
            next_episode_states = None
            active_players = set(states)
            rewards_by_player = {pid: 0.0 for pid in states}
            statistics_by_player = {}
            episode_steps_by_player = {pid: 0 for pid in states}
            behaviour_by_player = {
                pid: {
                    "actions": [0] * cfg["action_size"],
                    "exploratory": 0, "greedy": 0,
                    "q_sum": 0.0, "q_squared_sum": 0.0, "q_count": 0,
                    "q_min": float("inf"), "q_max": float("-inf"),
                }
                for pid in states
            }
            losses, td_errors = [], []

            for timestep in range(1, cfg["max_steps_per_episode"] + 1):
                if not active_players:
                    break

                ordered_players = sorted(active_players)
                actions = agent.choose_actions({pid: states[pid] for pid in ordered_players})

                for pid in ordered_players:
                    sel = agent.last_selection_by_player[pid]
                    beh = behaviour_by_player[pid]
                    beh["actions"][actions[pid]] += 1
                    beh["exploratory"] += int(sel["exploratory"])
                    beh["greedy"] += int(not sel["exploratory"])
                    beh["q_sum"] += sel["q_sum"]
                    beh["q_squared_sum"] += sel["q_squared_sum"]
                    beh["q_count"] += sel["q_count"]
                    beh["q_min"] = min(beh["q_min"], sel["q_min"])
                    beh["q_max"] = max(beh["q_max"], sel["q_max"])

                #Remove later
                #_t0 = time.perf_counter()
                responses, _ = await env.step(actions)
                #prof_times["godot_wait"] += time.perf_counter() - _t0
                #prof_counts["steps"] += 1

                reached_step_limit = (timestep == cfg["max_steps_per_episode"])

                for pid in ordered_players:
                    response = responses[pid]
                    reward = response["reward"]
                    done = response["done"]

                    if reached_step_limit and not done:
                        reward += cfg["timeout_penalty"]
                        done = True
                        response["done"] = True
                        response.setdefault("statistics", {})["death_cause"] = "timeout"

                    agent.remember(states[pid], actions[pid], reward, response["state"], done)
                    states[pid] = response["state"]
                    rewards_by_player[pid] += reward
                    statistics_by_player[pid] = response.get("statistics", {})
                    episode_steps_by_player[pid] += 1

                    if done:
                        active_players.remove(pid)

                for _ in range(cfg["training_updates_per_timestep"]):
                    #Remove later
                    #_t0 = time.perf_counter()
                    loss = agent.train_step()
                    #prof_times["train_step"] += time.perf_counter() - _t0
                    #prof_counts["updates"] += 1

                    if loss is not None:
                        losses.append(loss)
                        td_errors.append(agent.last_train_metrics["mean_abs_td_error"])

            agent.decay_epsilon()
            average_loss = sum(losses) / len(losses) if losses else 0.0
            loss_std = (sum((l - average_loss) ** 2 for l in losses) / len(losses)) ** 0.5 if losses else 0.0
            loss_max = max(losses) if losses else 0.0
            td_error = sum(td_errors) / len(td_errors) if td_errors else 0.0
            training_seconds = time.perf_counter() - episode_started_at

            for pid in statistics_by_player:
                stats = statistics_by_player[pid]
                if not stats.get("completed", False) and not stats.get("death_cause"):
                    stats["death_cause"] = "timeout"

            for pid in sorted(statistics_by_player):
                stats = statistics_by_player[pid]
                apples = stats.get("apples_collected", 0)
                distance = stats.get("distance_travelled", 0.0)
                snails = stats.get("snails_avoided", 0)
                steps = stats.get("episode_length", stats.get("steps", episode_steps_by_player[pid]))
                reward = rewards_by_player[pid]
                agent.update_player_performance(pid, reward)
                beh = behaviour_by_player[pid]
                q_count = beh["q_count"]
                q_mean = beh["q_sum"] / q_count if q_count else 0.0
                q_var = max(0.0, beh["q_squared_sum"] / q_count - q_mean ** 2) if q_count else 0.0

                # Per-player gameplay detail is still fully written to the statistics CSV every
                # episode regardless of console_log_frequency -- this print is purely a console
                # convenience and only fires on the throttled cadence to avoid flooding the output.
                if episode % cfg.get("console_log_frequency", 1) == 0:
                    print(
                        f"  P{pid} | Apples: {apples} | Distance: {distance:.2f} | "
                        f"Snails Avoided: {snails} | Steps: {steps}"
                    )

                statistics.add_episode(
                    episode=episode, player_id=pid, reward=reward,
                    score=stats.get("score", 0), loss=average_loss,
                    loss_std=loss_std, loss_max=loss_max, td_error=td_error,
                    epsilon=agent.epsilon, training_seconds=training_seconds,
                    episode_length=steps, distance_travelled=distance,
                    apples_collected=apples,
                    apple_collection_rate=stats.get("apple_collection_rate", 0.0),
                    snails_avoided=snails, completed=stats.get("completed", False),
                    death_cause=stats.get("death_cause", ""),
                    exploratory_actions=beh["exploratory"], greedy_actions=beh["greedy"],
                    q_mean=q_mean, q_max=beh["q_max"] if q_count else 0.0,
                    q_min=beh["q_min"] if q_count else 0.0,
                    q_std=math.sqrt(q_var), action_counts=beh["actions"],
                )

            avg_reward = sum(rewards_by_player.values()) / len(rewards_by_player)
            reward_str = " | ".join(f"P{pid}: {r:.2f}" for pid, r in sorted(rewards_by_player.items()))
            if episode % cfg.get("console_log_frequency", 1) == 0:
                progress_bar = _format_progress_bar(episode, cfg["max_episodes"])
                print(
                    f"{progress_bar} | {reward_str} | "
                    f"Epsilon: {agent.epsilon:.3f} | Loss: {average_loss:.4f}"
                )

            summary["final_reward"] = avg_reward
            summary["best_reward"] = max(summary["best_reward"], avg_reward)
            summary["total_episodes"] = episode

            os.makedirs(os.path.dirname(cfg["model_path"]), exist_ok=True)
            if episode % cfg["save_frequency"] == 0 or episode == cfg["max_episodes"]:
                #_t0 = time.perf_counter()
                agent.save(cfg["model_path"], episode)
                #prof_times["checkpoint_save"] += time.perf_counter() - _t0

                print("Model saved.")

            # --- ADD: print a report every console_log_frequency episodes ---
            '''
            if episode % cfg.get("console_log_frequency", 1) == 0:
                total = sum(prof_times.values())
                print(f"  [PROFILE ep {episode}] " + " | ".join(
                    f"{k}: {v:.1f}s ({100*v/total:.0f}%)" for k, v in prof_times.items()
                ))
                if prof_counts["steps"]:
                    print(f"  [PROFILE] steps/sec: {prof_counts['steps']/total:.2f} | "
                          f"updates/sec: {prof_counts['updates']/total:.1f}")
                    print(f"  [DEBUG] replay_buffer size: {len(agent.replay_buffer)} / min_replay_size: {cfg['min_replay_size']}")
            '''
            next_episode_states = await env.reset(seed)
            agent.reset_exploration_commitments()

    except (asyncio.CancelledError, KeyboardInterrupt):
        print(f"\n[INTERRUPT] Training manually stopped at episode {current_ep}.")
        if current_ep > 0:
            os.makedirs(os.path.dirname(cfg["model_path"]), exist_ok=True)
            agent.save(cfg["model_path"], current_ep)
            print(f"Saved checkpoint for episode {current_ep} to {cfg['model_path']}.")
        raise
    finally:
        statistics.close()
        await env.close()
        print("Cleaned up WebSocket connections and statistics files.")


    return summary

print("run_training() function defined with interrupt safety.")


run_training() function defined with interrupt safety.


### Cell 9 — Multi-Experiment Automation Manager
- **`build_config(overrides)`:** merges an experiment's override dict onto `DEFAULT_CONFIG` and auto-derives `model_path`/`statistics_path` from the experiment `name` unless explicitly overridden — this is what keeps multiple experiments' outputs from colliding.
- **`run_all_experiments(experiment_list)`:** runs each experiment sequentially with `run_training`, prints a banner before each one (name/seed/LR/batch size/output paths — the seed is always visible here for traceability), and prints a final comparison table. If interrupted mid-experiment, it stops cleanly and still prints the table for whichever experiments *did* complete.


In [11]:
def build_config(overrides: dict) -> dict:
    """Merge overrides into DEFAULT_CONFIG and generate experiment output paths."""
    cfg = {**DEFAULT_CONFIG, **overrides}

    name = cfg.get("name", f"experiment_seed_{cfg['seed']}")
    cfg["name"] = name

    if "model_path" not in overrides:
        cfg["model_path"] = f"results/models/{name}.pth"

    if "statistics_path" not in overrides:
        cfg["statistics_path"] = f"results/data/{name}.csv"

    return cfg


async def run_all_experiments(experiment_list: list):
    """Run all experiment configurations sequentially with safe interruption."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Training Execution Device: {device}")

    all_summaries = []
    total = len(experiment_list)

    try:
        for i, overrides in enumerate(experiment_list, start=1):
            cfg = build_config(overrides)

            print("\n" + "=" * 60)
            print(f"STARTING EXPERIMENT {i}/{total}: {cfg['name']}")
            print(
                f"Seed: {cfg['seed']} | "
                f"LR: {cfg['learning_rate']} | "
                f"Batch: {cfg['batch_size']}"
            )
            print(f"Model Path: {cfg['model_path']}")
            print(f"Stats Path: {cfg['statistics_path']}")
            print("=" * 60 + "\n")

            try:
                summary = await run_training(cfg, device)

                # Make sure the summary always contains the fields
                # required by the experiment summary table.
                if summary is None:
                    summary = {}

                summary.setdefault("name", cfg["name"])
                summary.setdefault("seed", cfg.get("seed"))
                summary.setdefault("total_episodes", None)
                summary.setdefault("final_reward", None)
                summary.setdefault("best_reward", None)

                # Prefer the actual configuration name/seed.
                if summary["name"] is None:
                    summary["name"] = cfg["name"]

                if summary["seed"] is None:
                    summary["seed"] = cfg.get("seed")

                all_summaries.append(summary)

                final_reward = summary.get("final_reward")
                best_reward = summary.get("best_reward")

                final_text = (
                    f"{float(final_reward):.2f}"
                    if final_reward is not None
                    else "N/A"
                )

                best_text = (
                    f"{float(best_reward):.2f}"
                    if best_reward is not None
                    and best_reward != float("-inf")
                    else "N/A"
                )

                print(
                    f"\n[COMPLETED] {cfg['name']} -> "
                    f"Final Avg Reward: {final_text} | "
                    f"Best: {best_text}"
                )

            except (asyncio.CancelledError, KeyboardInterrupt):
                print(
                    f"\n[INTERRUPT] Stopped multi-experiment "
                    f"execution during '{cfg['name']}'."
                )
                break

    finally:
        if all_summaries:
            print("\n" + "=" * 75)

            print(
                f"{'Experiment Name':<30} "
                f"{'Seed':>6} "
                f"{'Episodes':>10} "
                f"{'Final Reward':>14} "
                f"{'Best Reward':>12}"
            )

            print("-" * 75)

            for s in all_summaries:
                name = str(s.get("name", "Unknown"))

                seed = s.get("seed")
                seed_text = (
                    str(seed)
                    if seed is not None
                    else "N/A"
                )

                total_episodes = s.get("total_episodes")
                episodes_text = (
                    str(total_episodes)
                    if total_episodes is not None
                    else "N/A"
                )

                final_reward = s.get("final_reward")
                final_text = (
                    f"{float(final_reward):.2f}"
                    if final_reward is not None
                    else "N/A"
                )

                best_reward = s.get("best_reward")
                best_text = (
                    f"{float(best_reward):.2f}"
                    if best_reward is not None
                    and best_reward != float("-inf")
                    else "N/A"
                )

                print(
                    f"{name:<30} "
                    f"{seed_text:>6} "
                    f"{episodes_text:>10} "
                    f"{final_text:>14} "
                    f"{best_text:>12}"
                )

            print("=" * 75)

        else:
            print("\nNo experiments completed.")

    return all_summaries


print("Multi-Experiment Automation Manager defined (Interrupt Safe).")

Multi-Experiment Automation Manager defined (Interrupt Safe).


### Cell 10 — Define Experiments List
Edit this list to configure what gets trained when Cell 11 runs. Each dict is a set of overrides merged onto `DEFAULT_CONFIG` by `build_config` — you only ever need to include the keys you want to change from the defaults, though the 12 experiments below happen to specify every tunable hyperparameter explicitly for clarity. Running several entries here is how you get a fair multi-seed / multi-hyperparameter comparison in Cells 12–14.

**The 12 configured experiments** span a range of hyperparameter trade-offs, all starting from `seed=42` (except `chaos`, which uses `seed=123` to also test cross-seed variance):

| Experiment | What it varies |
|---|---|
| `baseline_seed42` | Default `DEFAULT_CONFIG` hyperparameters, unchanged |
| `eager_learner` | Higher learning rate (`1e-3`) + 8 updates/step (vs. 4) — learns aggressively |
| `megamind` | Larger network: 256 hidden units, 3 layers (vs. 128/2) |
| `nerd` | Bigger replay buffer (500k) + 8 updates/step + slower epsilon decay, 2,000 episodes |
| `eager_nerd` | `eager_learner`'s LR/updates combined with `nerd`'s buffer/schedule/episode count |
| `dementia` | Intentionally weak: tiny network/buffer, 1 update/step, fast epsilon decay — a baseline for "how bad can it get" |
| `explorer` | Epsilon floor raised to 0.15 (vs. 0.05) — keeps exploring instead of committing early |
| `patient` | Low learning rate (`1e-4`), only 2 updates/step, 2,000 episodes — slow and conservative |
| `speedrunner` | Small network/buffer/batch, few updates — prioritises wall-clock training speed |
| `tank` | Large network (256/2), low LR, big batch (128), 2,000 episodes — stable and conservative |
| `chaos` | Epsilon floor raised to 0.20 and a different seed (`123`) — never fully commits to greedy play |
| `marathon` | 5,000-episode run with a large buffer and slow epsilon decay — long-horizon training |


In [25]:
experiments = [
    {
        "name": "baseline_seed42",  # Baseline
        "learning_rate": 0.0003,
        "gamma": 0.99,
        "hidden_size": 128,
        "hidden_layers": 2,
        "replay_buffer_size": 300_000,
        "batch_size": 64,
        "min_replay_size": 2_000,
        "training_updates_per_timestep": 4,
        "epsilon_start": 1.0,
        "epsilon_end": 0.05,
        "epsilon_decay": 0.995,
        "target_update_frequency": 2_000,
        "max_episodes": 1_000,
        "seed": 42,
    },
    {
        "name": "eager_learner",  # Eager learner -- higher LR, more updates per step
        "learning_rate": 0.001,
        "gamma": 0.99,
        "hidden_size": 128,
        "hidden_layers": 2,
        "replay_buffer_size": 300_000,
        "batch_size": 64,
        "min_replay_size": 2_000,
        "training_updates_per_timestep": 8,
        "epsilon_start": 1.0,
        "epsilon_end": 0.05,
        "epsilon_decay": 0.995,
        "target_update_frequency": 2_000,
        "max_episodes": 1_000,
        "seed": 42,
    },
    {
        "name": "megamind",  # Megamind -- larger network (256 hidden, 3 layers)
        "learning_rate": 0.0003,
        "gamma": 0.99,
        "hidden_size": 256,
        "hidden_layers": 3,
        "replay_buffer_size": 300_000,
        "batch_size": 64,
        "min_replay_size": 2_000,
        "training_updates_per_timestep": 4,
        "epsilon_start": 1.0,
        "epsilon_end": 0.05,
        "epsilon_decay": 0.995,
        "target_update_frequency": 2_000,
        "max_episodes": 1_000,
        "seed": 42,
    },
    {
        "name": "nerd",  # Nerd -- big buffer, many updates, slow epsilon decay, long run
        "learning_rate": 0.0003,
        "gamma": 0.99,
        "hidden_size": 128,
        "hidden_layers": 2,
        "replay_buffer_size": 500_000,
        "batch_size": 64,
        "min_replay_size": 5_000,
        "training_updates_per_timestep": 8,
        "epsilon_start": 1.0,
        "epsilon_end": 0.03,
        "epsilon_decay": 0.998,
        "target_update_frequency": 4_000,
        "max_episodes": 2_000,
        "seed": 42,
    },
    {
        "name": "eager_nerd",  # Eager nerd -- eager_learner's LR/updates + nerd's buffer/schedule
        "learning_rate": 0.001,
        "gamma": 0.99,
        "hidden_size": 128,
        "hidden_layers": 2,
        "replay_buffer_size": 500_000,
        "batch_size": 64,
        "min_replay_size": 5_000,
        "training_updates_per_timestep": 8,
        "epsilon_start": 1.0,
        "epsilon_end": 0.03,
        "epsilon_decay": 0.998,
        "target_update_frequency": 4_000,
        "max_episodes": 2_000,
        "seed": 42,
    },
    {
        "name": "dementia",  # Dementia -- intentionally weak: tiny buffer/network, fast epsilon decay
        "learning_rate": 0.0003,
        "gamma": 0.95,
        "hidden_size": 64,
        "hidden_layers": 1,
        "replay_buffer_size": 20_000,
        "batch_size": 32,
        "min_replay_size": 500,
        "training_updates_per_timestep": 1,
        "epsilon_start": 1.0,
        "epsilon_end": 0.05,
        "epsilon_decay": 0.97,
        "target_update_frequency": 500,
        "max_episodes": 1_000,
        "seed": 42,
    },
    {
        "name": "explorer",  # Explorer -- keeps exploring longer (epsilon floor 0.15)
        "learning_rate": 0.0003,
        "gamma": 0.99,
        "hidden_size": 128,
        "hidden_layers": 2,
        "replay_buffer_size": 300_000,
        "batch_size": 64,
        "min_replay_size": 2_000,
        "training_updates_per_timestep": 4,
        "epsilon_start": 1.0,
        "epsilon_end": 0.15,
        "epsilon_decay": 0.998,
        "target_update_frequency": 2_000,
        "max_episodes": 1_000,
        "seed": 42,
    },
    {
        "name": "patient",  # Patient -- low LR, few updates per step, gradual learning
        "learning_rate": 0.0001,
        "gamma": 0.99,
        "hidden_size": 128,
        "hidden_layers": 2,
        "replay_buffer_size": 300_000,
        "batch_size": 64,
        "min_replay_size": 5_000,
        "training_updates_per_timestep": 2,
        "epsilon_start": 1.0,
        "epsilon_end": 0.05,
        "epsilon_decay": 0.998,
        "target_update_frequency": 4_000,
        "max_episodes": 2_000,
        "seed": 42,
    },
    {
        "name": "speedrunner",  # Speedrunner -- small/fast config prioritising wall-clock speed
        "learning_rate": 0.0005,
        "gamma": 0.99,
        "hidden_size": 64,
        "hidden_layers": 2,
        "replay_buffer_size": 50_000,
        "batch_size": 32,
        "min_replay_size": 1_000,
        "training_updates_per_timestep": 2,
        "epsilon_start": 1.0,
        "epsilon_end": 0.05,
        "epsilon_decay": 0.99,
        "target_update_frequency": 1_000,
        "max_episodes": 1_000,
        "seed": 42,
    },
    {
        "name": "tank",  # Tank -- large network, low LR, big batch, stable/conservative
        "learning_rate": 0.0001,
        "gamma": 0.99,
        "hidden_size": 256,
        "hidden_layers": 2,
        "replay_buffer_size": 500_000,
        "batch_size": 128,
        "min_replay_size": 5_000,
        "training_updates_per_timestep": 2,
        "epsilon_start": 1.0,
        "epsilon_end": 0.05,
        "epsilon_decay": 0.997,
        "target_update_frequency": 1_000,
        "max_episodes": 2_000,
        "seed": 42,
    },
    {
        "name": "chaos",  # Chaos -- high epsilon floor (0.20) and a different seed (123)
        "learning_rate": 0.0003,
        "gamma": 0.99,
        "hidden_size": 128,
        "hidden_layers": 2,
        "replay_buffer_size": 300_000,
        "batch_size": 64,
        "min_replay_size": 2_000,
        "training_updates_per_timestep": 4,
        "epsilon_start": 1.0,
        "epsilon_end": 0.2,
        "epsilon_decay": 0.995,
        "target_update_frequency": 2_000,
        "max_episodes": 1_000,
        "seed": 123,
    },
    {
        "name": "marathon",  # Marathon -- 5,000 episodes, large buffer, slow epsilon decay
        "learning_rate": 0.0003,
        "gamma": 0.99,
        "hidden_size": 128,
        "hidden_layers": 2,
        "replay_buffer_size": 500_000,
        "batch_size": 64,
        "min_replay_size": 5_000,
        "training_updates_per_timestep": 4,
        "epsilon_start": 1.0,
        "epsilon_end": 0.03,
        "epsilon_decay": 0.998,
        "target_update_frequency": 4_000,
        "max_episodes": 5_000,
        "seed": 42,
    },
]

print(f"Configured {len(experiments)} experiments:")
for i, exp in enumerate(experiments, 1):
    cfg = build_config(exp)
    print(
        f"  [{i:2d}] {cfg['name']:<16} | seed={cfg['seed']:<4} | lr={cfg['learning_rate']} | "
        f"hidden={cfg['hidden_size']}x{cfg['hidden_layers']} | batch={cfg['batch_size']} | "
        f"buffer={cfg['replay_buffer_size']:_} | eps_end={cfg['epsilon_end']} | "
        f"episodes={cfg['max_episodes']:_}"
    )


Configured 12 experiments:
  [ 1] baseline_seed42  | seed=42   | lr=0.0003 | hidden=128x2 | batch=64 | buffer=300_000 | eps_end=0.05 | episodes=1_000
  [ 2] eager_learner    | seed=42   | lr=0.001 | hidden=128x2 | batch=64 | buffer=300_000 | eps_end=0.05 | episodes=1_000
  [ 3] megamind         | seed=42   | lr=0.0003 | hidden=256x3 | batch=64 | buffer=300_000 | eps_end=0.05 | episodes=1_000
  [ 4] nerd             | seed=42   | lr=0.0003 | hidden=128x2 | batch=64 | buffer=500_000 | eps_end=0.03 | episodes=2_000
  [ 5] eager_nerd       | seed=42   | lr=0.001 | hidden=128x2 | batch=64 | buffer=500_000 | eps_end=0.03 | episodes=2_000
  [ 6] dementia         | seed=42   | lr=0.0003 | hidden=64x1 | batch=32 | buffer=20_000 | eps_end=0.05 | episodes=1_000
  [ 7] explorer         | seed=42   | lr=0.0003 | hidden=128x2 | batch=64 | buffer=300_000 | eps_end=0.15 | episodes=1_000
  [ 8] patient          | seed=42   | lr=0.0001 | hidden=128x2 | batch=64 | buffer=300_000 | eps_end=0.05 | episodes

### Cell 11 — Run All Training Experiments
**Execute this cell to start sequential training across all configured experiments.** This is a top-level `await` running inside Jupyter's existing asyncio event loop. To stop training, interrupt the kernel — see Cell 8's documentation for exactly what happens on interrupt.


In [14]:
# Top-level 'await' runs async coroutines within Jupyter's event loop.
results = await run_all_experiments(experiments)

Training Execution Device: cuda

STARTING EXPERIMENT 1/1: eager_learner
Seed: 42 | LR: 0.001 | Batch: 64
Model Path: results/models/eager_learner.pth
Stats Path: results/data/eager_learner.csv

Connected to Godot environment!
  P0 | Apples: 0 | Distance: 2982.98 | Snails Avoided: 1 | Steps: 86
  P1 | Apples: 0 | Distance: 5670.30 | Snails Avoided: 1 | Steps: 157
  P2 | Apples: 0 | Distance: 1554.20 | Snails Avoided: 0 | Steps: 35
  P3 | Apples: 0 | Distance: 3187.16 | Snails Avoided: 1 | Steps: 109
[#-----------------------]   2.5% (25/1000) | P0: -1.49 | P1: -2.03 | P2: -2.55 | P3: 2.38 | Epsilon: 0.882 | Loss: 0.3404
  P0 | Apples: 1 | Distance: 4229.55 | Snails Avoided: 0 | Steps: 125
  P1 | Apples: 0 | Distance: 1855.88 | Snails Avoided: 0 | Steps: 41
  P2 | Apples: 0 | Distance: 3005.94 | Snails Avoided: 0 | Steps: 72
  P3 | Apples: 0 | Distance: 1932.61 | Snails Avoided: 0 | Steps: 58
[#-----------------------]   5.0% (50/1000) | P0: -1.84 | P1: -1.53 | P2: -1.60 | P3: -3.58 | Ep

### Cell 12 — Automated Model Statistics Analysis & Tables
`analyze_all_models(experiment_list)` reads each experiment's statistics CSV directly from disk — it does not require training to still be running, or even the same kernel session — and prints a per-model report: episode count, training duration, reward mean/std/range, a 25-episode trailing moving-average reward, level completion / snail-death / timeout rates, apple-collection breakdown, and average loss/Q-value. This is the text-report counterpart to the graphs generated in Cells 13/14.


In [16]:
def analyze_all_models(experiment_list):
    """Parses training CSV logs for all models and prints a comprehensive statistical report."""
    reports = {}
    for overrides in experiment_list:
        cfg = build_config(overrides)
        name = cfg["name"]
        stats_path = cfg["statistics_path"]
        if not os.path.exists(stats_path):
            print(f"[Skip] Statistics file not found for '{name}': {stats_path}")
            continue

        df = pd.read_csv(stats_path)
        # Group by episode to get per-episode averages across parallel players
        ep_df = df.groupby("episode").agg({
            "reward": "mean",
            "score": "mean",
            "apples_collected": "mean",
            "snails_avoided": "mean",
            "completed": "mean",
            "loss": "mean",
            "q_mean": "mean",
            "training_seconds": "first"
        }).reset_index()

        total_episodes = len(ep_df)
        avg_reward = ep_df["reward"].mean()
        std_reward = ep_df["reward"].std()
        max_reward = ep_df["reward"].max()
        min_reward = ep_df["reward"].min()
        final_ma_reward = ep_df["reward"].tail(25).mean() if total_episodes >= 25 else avg_reward

        apple_counts = df["apples_collected"].value_counts(normalize=True) * 100
        pct_0 = apple_counts.get(0, 0.0)
        pct_1 = apple_counts.get(1, 0.0)
        pct_2 = apple_counts.get(2, 0.0)

        total_snails = df["snails_avoided"].sum()
        avg_snails = ep_df["snails_avoided"].mean()

        causes = df["death_cause"].fillna("").replace("", "completed" if df["completed"].any() else "unknown")
        cause_pcts = causes.value_counts(normalize=True) * 100
        completed_pct = (df["completed"].astype(bool)).mean() * 100
        snail_death_pct = cause_pcts.get("snail", 0.0)
        timeout_pct = cause_pcts.get("timeout", 0.0)

        avg_loss = ep_df["loss"].mean()
        avg_q = ep_df["q_mean"].mean()
        total_time_sec = ep_df["training_seconds"].sum()

        reports[name] = {
            "name": name,
            "total_episodes": total_episodes,
            "avg_reward": avg_reward,
            "std_reward": std_reward,
            "max_reward": max_reward,
            "min_reward": min_reward,
            "final_ma_reward": final_ma_reward,
            "pct_0_apples": pct_0,
            "pct_1_apple": pct_1,
            "pct_2_apples": pct_2,
            "total_snails": total_snails,
            "avg_snails": avg_snails,
            "completed_pct": completed_pct,
            "snail_death_pct": snail_death_pct,
            "timeout_pct": timeout_pct,
            "avg_loss": avg_loss,
            "avg_q": avg_q,
            "total_time_min": total_time_sec / 60.0
        }

        print("\n" + "=" * 65)
        print(f"MODEL ANALYSIS: {name}")
        print("=" * 65)
        print(f"Episodes Processed:   {total_episodes}")
        print(f"Training Duration:    {total_time_sec / 60.0:.2f} minutes")
        print(f"Average Reward:       {avg_reward:.2f} (±{std_reward:.2f})")
        print(f"Reward Range:         [{min_reward:.2f}, {max_reward:.2f}]")
        print(f"Final 25-Ep MA Reward: {final_ma_reward:.2f}")
        print(f"Level Completion Rate:{completed_pct:.2f}%")
        print(f"Snail Death Rate:     {snail_death_pct:.2f}%")
        print(f"Timeout Rate:         {timeout_pct:.2f}%")
        print(f"Apple Collection:     0 apples={pct_0:.1f}% | 1 apple={pct_1:.1f}% | 2 apples={pct_2:.1f}%")
        print(f"Snails Avoided:       {total_snails:.0f} total (avg {avg_snails:.2f}/ep)")
        print(f"Average Loss / Q:     Loss={avg_loss:.4f} | Q_mean={avg_q:.2f}")
        print("=" * 65)

    return reports

analysis_reports = analyze_all_models(experiments)


MODEL ANALYSIS: baseline_seed42
Episodes Processed:   1000
Training Duration:    48.81 minutes
Average Reward:       19.36 (±23.64)
Reward Range:         [-4.33, 98.53]
Final 25-Ep MA Reward: 49.56
Level Completion Rate:16.57%
Snail Death Rate:     82.70%
Timeout Rate:         0.73%
Apple Collection:     0 apples=49.6% | 1 apple=33.8% | 2 apples=16.6%
Snails Avoided:       2649 total (avg 0.66/ep)
Average Loss / Q:     Loss=0.0876 | Q_mean=31.53

MODEL ANALYSIS: eager_learner
Episodes Processed:   1000
Training Duration:    82.15 minutes
Average Reward:       6.77 (±11.08)
Reward Range:         [-7.05, 79.83]
Final 25-Ep MA Reward: 9.12
Level Completion Rate:3.25%
Snail Death Rate:     95.62%
Timeout Rate:         1.12%
Apple Collection:     0 apples=57.1% | 1 apple=39.6% | 2 apples=3.2%
Snails Avoided:       3112 total (avg 0.78/ep)
Average Loss / Q:     Loss=3.1959 | Q_mean=1193.35

MODEL ANALYSIS: megamind
Episodes Processed:   1000
Training Duration:    44.94 minutes
Average Rewar

### Modular Plotting Library
This is the **major requirement #9/#10 implementation**: every graph is its own small, independently testable `plot_*(name, ep_df, df, model_dir, window)` function, switched on/off from the single `PLOT_CONFIG` dict below. Adding a graph = write one function + add one `PLOT_CONFIG` entry + add one call in `generate_individual_plots` (Cell 13). Removing a graph = flip its `PLOT_CONFIG` value to `False`; no other code changes anywhere.

All plots are computed **after** training from the saved CSV — nothing here runs during the training loop, so graph generation never slows down or interferes with training.

Covers every graph requested in the brief that the existing statistics can actually support: total reward, learning curve (raw + moving average, window labelled), episode length, score, apples collected, snails avoided, distance travelled, epsilon decay, training loss, average Q-value, an approximate Q-value spread (mean ± std band — a true per-action Q-value histogram isn't available since only per-episode aggregates are logged), rolling completion rate, action distribution (overall and early-vs-late), episode end causes, apple distribution, and per-player consistency. `learning_rate` is registered but disabled by default since this notebook uses a fixed Adam learning rate with no scheduler — there's nothing meaningful to plot until one is added. Reward-component breakdown (apple/completion/death/step-penalty/etc.) is **not** included because the environment only reports a single combined `reward` value per step, not per-component rewards — adding that graph would mean inventing data the statistics don't contain.


In [13]:
PLOT_CONFIG = {
    # Core per-episode trend graphs
    "total_reward": True,
    "learning_curve": True,
    "episode_length": True,
    "score": True,
    "apples_collected": True,
    "snails_avoided": True,
    "distance_travelled": True,

    # DDQN training diagnostics
    "epsilon_decay": True,
    "loss": True,
    "q_values": True,
    "q_value_distribution": True,
    "learning_rate": False,  # disabled: this notebook uses a fixed Adam LR (no scheduler), so there is nothing to plot
    "completion_rate": True,

    # Gameplay / behaviour breakdowns
    "action_distribution": True,
    "action_distribution_early_late": True,
    "end_causes": True,
    "apple_distribution": True,
    "player_consistency": True,
}
# To add a new graph: write a plot_<name>(name, ep_df, df, model_dir, window) function below,
# add one entry to PLOT_CONFIG, and one line inside generate_individual_plots(). To remove a
# graph from the output, just flip its PLOT_CONFIG value to False -- no other code changes needed.


def _save(fig, model_dir, filename, dpi=200):
    fig.tight_layout()
    fig.savefig(model_dir / filename, dpi=dpi)
    plt.close(fig)


def plot_total_reward(name, ep_df, df, model_dir, window):
    """Episode number vs. total (raw, per-episode-mean-across-players) reward."""
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ep_df.episode, ep_df.reward, color="#1f77b4", alpha=0.6)
    ax.set(xlabel="Episode", ylabel="Reward", title=f"{name} — Total Reward per Episode")
    ax.grid(alpha=0.3)
    _save(fig, model_dir, "total_reward.png")


def plot_learning_curve(name, ep_df, df, model_dir, window):
    """Raw reward plus a clearly-labelled rolling moving average -- the standard RL learning curve."""
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ep_df.episode, ep_df.reward, alpha=0.3, label="Raw episode reward")
    ax.plot(ep_df.episode, ep_df.reward_ma, linewidth=2, color="#1f77b4",
            label=f"{window}-episode moving average")
    ax.set(xlabel="Episode", ylabel="Reward", title=f"{name} — Learning Curve")
    ax.grid(alpha=0.3)
    ax.legend()
    _save(fig, model_dir, "learning_curve.png")


def plot_episode_length(name, ep_df, df, model_dir, window):
    """Episode number vs. number of steps taken."""
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ep_df.episode, ep_df.episode_length, color="#4363D8", alpha=0.7)
    ax.set(xlabel="Episode", ylabel="Steps", title=f"{name} — Episode Length")
    ax.grid(alpha=0.3)
    _save(fig, model_dir, "episode_length.png")


def plot_score(name, ep_df, df, model_dir, window):
    """Episode number vs. in-game score (tracks apple collection / level progress)."""
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ep_df.episode, ep_df.score, color="#F58231", alpha=0.7)
    ax.set(xlabel="Episode", ylabel="Score", title=f"{name} — Score per Episode")
    ax.grid(alpha=0.3)
    _save(fig, model_dir, "score.png")


def plot_apples_collected(name, ep_df, df, model_dir, window):
    """Episode number vs. apples collected, with a rolling moving average."""
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ep_df.episode, ep_df.apples_collected, alpha=0.3, color="#3CB44B")
    ax.plot(ep_df.episode, ep_df.apples_ma, linewidth=2, color="#2a8f3a",
            label=f"{window}-episode moving average")
    ax.set(xlabel="Episode", ylabel="Apples Collected", title=f"{name} — Apples Collected per Episode")
    ax.grid(alpha=0.3)
    ax.legend()
    _save(fig, model_dir, "apples_collected.png")


def plot_snails_avoided(name, ep_df, df, model_dir, window):
    """Episode number vs. snail enemies avoided."""
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ep_df.episode, ep_df.snails_avoided, color="#911EB4", alpha=0.7)
    ax.set(xlabel="Episode", ylabel="Snails Avoided", title=f"{name} — Snail Enemies Avoided per Episode")
    ax.grid(alpha=0.3)
    _save(fig, model_dir, "snails_avoided.png")


def plot_distance_travelled(name, ep_df, df, model_dir, window):
    """Episode number vs. distance travelled."""
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ep_df.episode, ep_df.distance_travelled, color="#469990", alpha=0.7)
    ax.set(xlabel="Episode", ylabel="Distance Travelled", title=f"{name} — Distance Travelled per Episode")
    ax.grid(alpha=0.3)
    _save(fig, model_dir, "distance_travelled.png")


def plot_epsilon_decay(name, ep_df, df, model_dir, window):
    """Episode number vs. exploration epsilon."""
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ep_df.episode, ep_df.epsilon, color="#E6194B")
    ax.set(xlabel="Episode", ylabel="Epsilon", title=f"{name} — Epsilon Decay")
    ax.grid(alpha=0.3)
    _save(fig, model_dir, "epsilon_decay.png")


def plot_loss(name, ep_df, df, model_dir, window):
    """Episode number vs. average training loss, with a rolling moving average."""
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ep_df.episode, ep_df.loss, alpha=0.3, color="orange")
    ax.plot(ep_df.episode, ep_df.loss_ma, linewidth=2, color="darkorange",
            label=f"{window}-episode moving average")
    ax.set(xlabel="Episode", ylabel="Loss", title=f"{name} — Training Loss")
    ax.grid(alpha=0.3)
    ax.legend()
    _save(fig, model_dir, "loss.png")


def plot_q_values(name, ep_df, df, model_dir, window):
    """Episode number vs. average Q-value, with a rolling moving average."""
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ep_df.episode, ep_df.q_mean, alpha=0.3, color="green")
    ax.plot(ep_df.episode, ep_df.q_ma, linewidth=2, color="forestgreen",
            label=f"{window}-episode moving average")
    ax.set(xlabel="Episode", ylabel="Mean Q-Value", title=f"{name} — Average Q-Value")
    ax.grid(alpha=0.3)
    ax.legend()
    _save(fig, model_dir, "q_values.png")


def plot_q_value_distribution(name, ep_df, df, model_dir, window):
    """Approximate Q-value spread over training: per-episode mean +/- std as a shaded band.

    The raw per-action-selection Q-values aren't individually retained (only per-episode
    aggregates: q_mean/q_std/q_min/q_max are logged), so a true value histogram isn't available.
    This mean +/- 1 std band is the most informative distribution view the existing statistics
    support without inventing data.
    """
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ep_df.episode, ep_df.q_mean, color="forestgreen", linewidth=1.5, label="Mean Q-value")
    ax.fill_between(ep_df.episode, ep_df.q_mean - ep_df.q_std, ep_df.q_mean + ep_df.q_std,
                     color="forestgreen", alpha=0.2, label="±1 std (per-episode)")
    ax.set(xlabel="Episode", ylabel="Q-Value", title=f"{name} — Q-Value Spread")
    ax.grid(alpha=0.3)
    ax.legend()
    _save(fig, model_dir, "q_value_distribution.png")


def plot_completion_rate(name, ep_df, df, model_dir, window):
    """Rolling level-completion rate over training."""
    completion_ma = ep_df["completed"].rolling(window, min_periods=1).mean() * 100
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(ep_df.episode, completion_ma, color="#3CB44B", linewidth=2)
    ax.set(xlabel="Episode", ylabel="Completion Rate (%)",
           title=f"{name} — {window}-Episode Rolling Completion Rate")
    ax.set_ylim(0, 105)
    ax.grid(alpha=0.3)
    _save(fig, model_dir, "completion_rate.png")


def plot_action_distribution(name, ep_df, df, model_dir, window, action_cols):
    """Overall count of each action selected across the whole run."""
    avail_cols = [c for c in action_cols if c in df.columns]
    if len(avail_cols) != 6:
        return
    labels = [c.replace("action_", "").replace("_", "+") for c in avail_cols]
    counts = ep_df[avail_cols].sum().values
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(labels, counts, color="#4363D8")
    ax.set(ylabel="Total Count", title=f"{name} — Overall Action Distribution")
    _save(fig, model_dir, "action_distribution_overall.png")


def plot_action_distribution_early_late(name, ep_df, df, model_dir, window, action_cols):
    """Compares action-selection mix in the first vs. last `window` episodes."""
    avail_cols = [c for c in action_cols if c in df.columns]
    if len(avail_cols) != 6:
        return
    labels = [c.replace("action_", "").replace("_", "+") for c in avail_cols]
    early_df = ep_df[ep_df.episode <= ep_df.episode.min() + window - 1][avail_cols].sum()
    late_df = ep_df[ep_df.episode >= ep_df.episode.max() - window + 1][avail_cols].sum()
    comp = pd.DataFrame({"Early Training": early_df, "Late Training": late_df}).T
    comp = comp.div(comp.sum(axis=1), axis=0) * 100
    comp.columns = labels
    fig, ax = plt.subplots(figsize=(9, 5))
    comp.plot(kind="bar", stacked=True, ax=ax, colormap="tab10")
    ax.set(ylabel="Percentage of Actions", title=f"{name} — Action Selection: Early vs Late")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    _save(fig, model_dir, "action_distribution_early_late.png")


def plot_end_causes(name, ep_df, df, model_dir, window):
    """Breakdown of how episodes ended (completion / death cause / timeout)."""
    causes = df["death_cause"].fillna("").replace("", "completed" if df["completed"].any() else "unknown")
    cause_counts = causes.value_counts()
    fig, ax = plt.subplots(figsize=(7, 4.5))
    cause_counts.plot(kind="bar", ax=ax, color=["#3CB44B", "#E6194B", "#F58231"])
    ax.set(xlabel="Termination Cause", ylabel="Player Episodes", title=f"{name} — Episode End Causes")
    _save(fig, model_dir, "end_causes.png")


def plot_apple_distribution(name, ep_df, df, model_dir, window):
    """Histogram of how many apples were collected per player-episode."""
    apple_dist = df["apples_collected"].value_counts(normalize=True).reindex([0, 1, 2], fill_value=0) * 100
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(["0 Apples", "1 Apple", "2 Apples"], apple_dist.values, color=["#E6194B", "#FFE119", "#3CB44B"])
    ax.set(ylabel="Percentage of Episodes", title=f"{name} — Apple Collection Breakdown")
    _save(fig, model_dir, "apple_distribution.png")


def plot_player_consistency(name, ep_df, df, model_dir, window):
    """Average reward broken down by player ID (P0-P3), to spot systematic per-slot bias."""
    player_perf = df.groupby("player_id")["reward"].mean()
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar([f"P{pid}" for pid in player_perf.index], player_perf.values, color="#911EB4")
    ax.set(xlabel="Player ID", ylabel="Average Reward", title=f"{name} — Player ID Consistency")
    _save(fig, model_dir, "player_consistency.png")


print(f"Modular plotting library loaded: {len(PLOT_CONFIG)} graph types registered "
      f"({sum(PLOT_CONFIG.values())} enabled).")


Modular plotting library loaded: 18 graph types registered (17 enabled).


### Cell 13 — Individual Model Graph Generation (Training & Gameplay Plots)
Generates every `PLOT_CONFIG`-enabled graph (see the modular plotting library above) for each model in `results/graphs/{name}/`. Re-run this cell any time after training (or across separate sessions) — it only reads each model's statistics CSV, so it never touches the environment, GPU, or checkpoints.

**Note on filenames vs. the pre-refactor version:** most filenames are unchanged (`action_distribution_*`, `end_causes.png`, `apple_distribution.png`, `player_consistency.png`). A few were split/renamed for clarity now that each graph is its own function: the combined reward file is now `learning_curve.png` (raw + moving average, same content as before) plus a new standalone `total_reward.png`; the combined loss+Q figure is now two files, `loss.png` and `q_values.png`. These are all regenerated outputs (not checkpoints or training state), so the renames don't affect reproducibility or resumability.


In [18]:
ACTION_NAMES = ("idle", "left", "right", "jump", "left_jump", "right_jump")
ACTION_COLS = [f"action_{n}" for n in ACTION_NAMES]


def generate_individual_plots(experiment_list, window=25):
    """Generates every PLOT_CONFIG-enabled per-model graph in results/graphs/{name}/.

    Reads each model's statistics CSV once, computes the shared per-episode aggregation +
    rolling-average columns every plot function needs, then calls each enabled plot_* function.
    Purely a consumer of PLOT_CONFIG -- to change which graphs are produced, edit PLOT_CONFIG above
    rather than this function.
    """
    for overrides in experiment_list:
        cfg = build_config(overrides)
        name = cfg["name"]
        stats_path = cfg["statistics_path"]
        if not os.path.exists(stats_path):
            print(f"[Skip] Statistics file not found for '{name}': {stats_path}")
            continue

        df = pd.read_csv(stats_path)
        model_dir = Path(f"results/graphs/{name}")
        model_dir.mkdir(parents=True, exist_ok=True)

        agg_dict = {
            "reward": "mean", "loss": "mean", "score": "mean",
            "apples_collected": "mean", "completed": "mean", "snails_avoided": "mean",
            "episode_length": "mean", "distance_travelled": "mean",
            "q_mean": "mean", "q_std": "mean", "epsilon": "mean",
        }
        for col in ACTION_COLS:
            if col in df.columns:
                agg_dict[col] = "sum"

        ep_df = df.groupby("episode").agg(agg_dict).reset_index()
        ep_df["reward_ma"] = ep_df["reward"].rolling(window, min_periods=1).mean()
        ep_df["loss_ma"] = ep_df["loss"].rolling(window, min_periods=1).mean()
        ep_df["apples_ma"] = ep_df["apples_collected"].rolling(window, min_periods=1).mean()
        ep_df["q_ma"] = ep_df["q_mean"].rolling(window, min_periods=1).mean()

        saved = 0
        if PLOT_CONFIG.get("total_reward"):
            plot_total_reward(name, ep_df, df, model_dir, window); saved += 1
        if PLOT_CONFIG.get("learning_curve"):
            plot_learning_curve(name, ep_df, df, model_dir, window); saved += 1
        if PLOT_CONFIG.get("episode_length"):
            plot_episode_length(name, ep_df, df, model_dir, window); saved += 1
        if PLOT_CONFIG.get("score"):
            plot_score(name, ep_df, df, model_dir, window); saved += 1
        if PLOT_CONFIG.get("apples_collected"):
            plot_apples_collected(name, ep_df, df, model_dir, window); saved += 1
        if PLOT_CONFIG.get("snails_avoided"):
            plot_snails_avoided(name, ep_df, df, model_dir, window); saved += 1
        if PLOT_CONFIG.get("distance_travelled"):
            plot_distance_travelled(name, ep_df, df, model_dir, window); saved += 1
        if PLOT_CONFIG.get("epsilon_decay"):
            plot_epsilon_decay(name, ep_df, df, model_dir, window); saved += 1
        if PLOT_CONFIG.get("loss"):
            plot_loss(name, ep_df, df, model_dir, window); saved += 1
        if PLOT_CONFIG.get("q_values"):
            plot_q_values(name, ep_df, df, model_dir, window); saved += 1
        if PLOT_CONFIG.get("q_value_distribution"):
            plot_q_value_distribution(name, ep_df, df, model_dir, window); saved += 1
        if PLOT_CONFIG.get("completion_rate"):
            plot_completion_rate(name, ep_df, df, model_dir, window); saved += 1
        if PLOT_CONFIG.get("action_distribution"):
            plot_action_distribution(name, ep_df, df, model_dir, window, ACTION_COLS); saved += 1
        if PLOT_CONFIG.get("action_distribution_early_late"):
            plot_action_distribution_early_late(name, ep_df, df, model_dir, window, ACTION_COLS); saved += 1
        if PLOT_CONFIG.get("end_causes"):
            plot_end_causes(name, ep_df, df, model_dir, window); saved += 1
        if PLOT_CONFIG.get("apple_distribution"):
            plot_apple_distribution(name, ep_df, df, model_dir, window); saved += 1
        if PLOT_CONFIG.get("player_consistency"):
            plot_player_consistency(name, ep_df, df, model_dir, window); saved += 1

        print(f"Saved {saved} plots for '{name}' in results/graphs/{name}/ "
              f"({len(PLOT_CONFIG) - saved} disabled via PLOT_CONFIG).")

generate_individual_plots(experiments)


Saved 17 plots for 'baseline_seed42' in results/graphs/baseline_seed42/ (1 disabled via PLOT_CONFIG).
Saved 17 plots for 'eager_learner' in results/graphs/eager_learner/ (1 disabled via PLOT_CONFIG).
Saved 17 plots for 'megamind' in results/graphs/megamind/ (1 disabled via PLOT_CONFIG).
Saved 17 plots for 'nerd' in results/graphs/nerd/ (1 disabled via PLOT_CONFIG).
Saved 17 plots for 'eager_nerd' in results/graphs/eager_nerd/ (1 disabled via PLOT_CONFIG).
Saved 17 plots for 'dementia' in results/graphs/dementia/ (1 disabled via PLOT_CONFIG).
Saved 17 plots for 'explorer' in results/graphs/explorer/ (1 disabled via PLOT_CONFIG).
Saved 17 plots for 'patient' in results/graphs/patient/ (1 disabled via PLOT_CONFIG).
Saved 17 plots for 'speedrunner' in results/graphs/speedrunner/ (1 disabled via PLOT_CONFIG).
Saved 17 plots for 'tank' in results/graphs/tank/ (1 disabled via PLOT_CONFIG).
Saved 17 plots for 'chaos' in results/graphs/chaos/ (1 disabled via PLOT_CONFIG).
Saved 17 plots for 'ma

### Cell 14 — Overall Comparative Model Plotting (Cross-Model Overlay Plots)
Compares all trained models on unified overlay plots in `results/graphs/overall/`: reward, loss, and apple-collection moving averages overlaid per model, plus a summary bar chart of average/best-moving-average reward and completion rate. Left structurally as-is (a single function) since, unlike the per-model diagnostic suite in Cell 13, there is a fixed, small set of cross-model comparisons rather than an open-ended list of graph types — the same `results/graphs/overall/` filenames are preserved. Purely reads statistics CSVs, so it can be re-run independently at any time.


In [20]:
def generate_overall_plots(experiment_list, window=25):
    """Generates comparative overlay graphs across all models in results/graphs/overall/."""
    overall_dir = Path("results/graphs/overall")
    overall_dir.mkdir(parents=True, exist_ok=True)

    model_dfs = {}
    for overrides in experiment_list:
        cfg = build_config(overrides)
        name = cfg["name"]
        stats_path = cfg["statistics_path"]
        if not os.path.exists(stats_path):
            continue

        df = pd.read_csv(stats_path)
        ep_df = df.groupby("episode").agg({
            "reward": "mean",
            "loss": "mean",
            "apples_collected": "mean",
            "completed": "mean",
            "snails_avoided": "mean"
        }).reset_index()
        ep_df["reward_ma"] = ep_df["reward"].rolling(window, min_periods=1).mean()
        ep_df["loss_ma"] = ep_df["loss"].rolling(window, min_periods=1).mean()
        ep_df["apples_ma"] = ep_df["apples_collected"].rolling(window, min_periods=1).mean()
        model_dfs[name] = ep_df

    if not model_dfs:
        print("No model data found for overall comparison.")
        return

    # 1. Comparative Reward Overlay
    fig, ax = plt.subplots(figsize=(10, 6))
    for name, ep_df in model_dfs.items():
        ax.plot(ep_df.episode, ep_df.reward_ma, linewidth=2, label=name)
    ax.set(xlabel="Episode", ylabel=f"{window}-Ep MA Reward", title="Overall Comparative Reward Trajectories")
    ax.grid(alpha=0.35)
    ax.legend()
    fig.tight_layout()
    fig.savefig(overall_dir / "comparative_rewards.png", dpi=300)
    plt.close(fig)

    # 2. Comparative Loss Overlay
    fig, ax = plt.subplots(figsize=(10, 6))
    for name, ep_df in model_dfs.items():
        ax.plot(ep_df.episode, ep_df.loss_ma, linewidth=2, label=name)
    ax.set(xlabel="Episode", ylabel=f"{window}-Ep MA Loss", title="Overall Comparative Training Loss")
    ax.grid(alpha=0.35)
    ax.legend()
    fig.tight_layout()
    fig.savefig(overall_dir / "comparative_losses.png", dpi=300)
    plt.close(fig)

    # 3. Comparative Apple Collection Overlay
    fig, ax = plt.subplots(figsize=(10, 6))
    for name, ep_df in model_dfs.items():
        ax.plot(ep_df.episode, ep_df.apples_ma, linewidth=2, label=name)
    ax.set(xlabel="Episode", ylabel=f"{window}-Ep MA Apples Collected", title="Overall Comparative Apple Collection Trajectories")
    ax.grid(alpha=0.35)
    ax.legend()
    fig.tight_layout()
    fig.savefig(overall_dir / "comparative_apple_collection.png", dpi=300)
    plt.close(fig)

    # 4. Comparative Summary Bar Chart
    summary_data = []
    for name, ep_df in model_dfs.items():
        summary_data.append({
            "model": name,
            "avg_reward": ep_df["reward"].mean(),
            "best_ma": ep_df["reward_ma"].max(),
            "completion_rate": ep_df["completed"].mean() * 100
        })
    sum_df = pd.DataFrame(summary_data)
    fig, ax = plt.subplots(figsize=(10, 6))
    sum_df.plot(x="model", y=["avg_reward", "best_ma"], kind="bar", ax=ax)
    ax.set(xlabel="Model", ylabel="Reward Metric", title="Overall Model Performance Comparison")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(overall_dir / "comparative_summary_bars.png", dpi=300)
    plt.close(fig)

    print(f"Saved 4 comparative plots in results/graphs/overall/")

generate_overall_plots(experiments)

Saved 4 comparative plots in results/graphs/overall/


### Cell 15 — Automated Greedy Validation Engine (10 Validation Episodes, Epsilon = 0)
Evaluates every trained model for `val_episodes` (default 10) with a **purely greedy policy** (`epsilon=0.0` — no exploration), completely separate from training: it never calls `agent.remember`, `agent.train_step`, or `agent.save`, so no replay-buffer contents, model weights, or training logs are ever touched or overwritten by validation.

**Validation seed:** each validation episode `ep` resets the environment with `cfg['seed'] + ep * 100`, so every validation episode gets its own distinct, reproducible level layout (unlike training, which reuses `cfg['seed']` for every episode by design — see the Reproducibility cell) — this provides a controlled benchmark of trained policy validation performance.

**Per-episode validation table:** written to `results/data/testing/{name}_test_results.csv` with one row per `(episode, player_id)`: episode, player_id, reward, apples_collected, completed, death_cause.

On completion (or on error, via the `finally` block), `close_godot()` is called to cleanly terminate the Godot process rather than leaving it orphaned.


In [14]:
async def run_evaluation_suite(experiment_list, episodes=10, mode="testing"):
    """
    Evaluates each trained model with epsilon=0 without modifying or saving weights.

    Args:
        experiment_list: List of experiment configurations.
        episodes: Number of evaluation episodes per model.
        mode: Defaults to "testing".

    Returns:
        Dictionary containing average reward and CSV path for each model.
    """

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    evaluation_dir = Path(f"results/data/{mode}")
    evaluation_dir.mkdir(parents=True, exist_ok=True)

    mode_title = mode.capitalize()

    env = None
    all_results = {}

    try:
        for overrides in experiment_list:
            cfg = build_config(overrides)
            name = cfg["name"]
            model_path = cfg["model_path"]

            if not os.path.exists(model_path):
                print(
                    f"[Skip {mode_title}] Model file not found for "
                    f"'{name}': {model_path}"
                )
                continue

            print("\n" + "=" * 60)
            print(
                f"{mode_title.upper()} MODEL: {name} "
                f"({episodes} Episodes, Epsilon = 0.0)"
            )
            print(f"Loading Model: {model_path}")
            print("=" * 60)

            agent = DQNAgent(cfg, device)
            agent.load(model_path)
            agent.policy_net.eval()

            if env is None:
                env = PlatformerEnvironment(cfg)
                await env.connect()

            csv_path = evaluation_dir / f"{name}_{mode}_results.csv"

            with open(
                csv_path,
                "w",
                newline="",
                encoding="utf-8"
            ) as f:
                writer = csv.writer(f)

                writer.writerow([
                    "episode",
                    "player_id",
                    "reward",
                    "apples_collected",
                    "completed",
                    "death_cause"
                ])

                episode_rewards = []
                completions = 0

                for ep in range(1, episodes + 1):
                    states = await env.reset(cfg["seed"] + ep * 100)
                    active_players = set(states)

                    ep_rewards = {
                        pid: 0.0
                        for pid in states
                    }

                    statistics_by_player = {}

                    for timestep in range(
                        1,
                        cfg["max_steps_per_episode"] + 1
                    ):
                        if not active_players:
                            break

                        ordered_players = sorted(active_players)

                        actions = agent.choose_actions(
                            {
                                pid: states[pid]
                                for pid in ordered_players
                            },
                            epsilon=0.0
                        )

                        responses, _ = await env.step(actions)

                        for pid in ordered_players:
                            res = responses[pid]

                            ep_rewards[pid] += res["reward"]
                            states[pid] = res["state"]

                            statistics_by_player[pid] = res.get(
                                "statistics",
                                {}
                            )

                            if res["done"]:
                                active_players.remove(pid)

                    avg_ep_reward = (
                        sum(ep_rewards.values()) / len(ep_rewards)
                    )

                    episode_rewards.append(avg_ep_reward)

                    for pid, reward in ep_rewards.items():
                        stats = statistics_by_player.get(pid, {})

                        completed = stats.get(
                            "completed",
                            False
                        )

                        if completed:
                            completions += 1

                        writer.writerow([
                            ep,
                            pid,
                            reward,
                            stats.get("apples_collected", 0),
                            completed,
                            stats.get("death_cause", "")
                        ])

                    reward_str = " | ".join(
                        f"P{p}: {r:.1f}"
                        for p, r in sorted(ep_rewards.items())
                    )

                    print(
                        f"{mode_title} Episode "
                        f"{ep:2d}/{episodes} | "
                        f"Avg Reward: {avg_ep_reward:6.2f} | "
                        f"Rewards: {reward_str}"
                    )

            avg_reward = (
                sum(episode_rewards) / len(episode_rewards)
            )

            all_results[name] = {
                "avg_reward": avg_reward,
                "csv": str(csv_path)
            }

            print(
                f"[{mode_title} Done] '{name}' -> "
                f"Average Greedy Reward: {avg_reward:.2f} | "
                f"Saved: {csv_path}"
            )

    finally:
        if env is not None:
            await env.close_godot()
            print("Godot engine process cleanly terminated.")

    return all_results

In [24]:
testing_results = await run_evaluation_suite(
    experiments,
    episodes=10,
    mode="testing"
)


TESTING MODEL: eager_nerd (10 Episodes, Epsilon = 0.0)
Loading Model: results/models/eager_nerd.pth
Connected to Godot environment!
Testing Episode  1/10 | Avg Reward:  60.20 | Rewards: P0: 81.3 | P1: 95.8 | P2: 81.4 | P3: 97.1 | P4: 81.5 | P5: 96.3 | P6: 98.2 | P7: 81.6 | P8: 81.5 | P9: -2.8 | P10: 96.3 | P11: 6.8 | P12: 82.0 | P13: 81.7 | P14: 98.6 | P15: 98.4 | P16: 97.0 | P17: 22.8 | P18: 81.8 | P19: 26.1 | P20: 96.0 | P21: 96.0 | P22: 81.9 | P23: -3.1 | P24: 99.1 | P25: 19.0 | P26: -2.2 | P27: 84.3 | P28: -2.9 | P29: -1.1 | P30: -3.5 | P31: 8.0 | P32: 84.6 | P33: 99.1 | P34: 2.9 | P35: 97.7 | P36: -1.6 | P37: 97.8 | P38: -3.2 | P39: 82.4 | P40: 97.9 | P41: 84.3 | P42: 84.3 | P43: 22.1 | P44: -2.8 | P45: 85.6 | P46: 100.3 | P47: 82.7 | P48: 7.8 | P49: -1.0 | P50: 97.1 | P51: 23.6 | P52: -0.8 | P53: 97.8 | P54: 82.3 | P55: 82.9 | P56: -1.9 | P57: 98.1 | P58: 22.8 | P59: 8.9 | P60: 82.9 | P61: 100.2 | P62: 100.4 | P63: 82.6
Testing Episode  2/10 | Avg Reward:  60.92 | Rewards: P0: 8

### Cell 16 — Automated Testing Result Graphing
Parses every `results/data/testing/{name}_test_results.csv` produced by Cell 15 and generates four evaluation-specific graphs in `results/graphs/testing/`, kept **entirely separate** from the training graphs in Cells 13/14 (different directory, different data source): mean greedy-test reward per model, level completion rate per model, per-episode reward trajectory across the 10 test episodes (one line per model), and average apples collected per model. These represent trained-policy evaluation performance, in contrast to Cell 13's `learning_curve.png`, which represents *training* progress.


In [14]:
def generate_plots(experiment_list, mode="testing"):
    """Generates graphs for testing evaluation results."""

    data_dir = Path(f"results/data/{mode}")
    graph_dir = Path(f"results/graphs/{mode}")
    graph_dir.mkdir(parents=True, exist_ok=True)

    model_test_dfs = {}

    for overrides in experiment_list:
        cfg = build_config(overrides)
        name = cfg["name"]

        csv_path = data_dir / f"{name}_{mode}_results.csv"

        if csv_path.exists():
            model_test_dfs[name] = pd.read_csv(csv_path)

    if not model_test_dfs:
        print(f"No {mode} data found in {data_dir}/ for plotting.")
        return

    summary = []

    for name, df in model_test_dfs.items():
        ep_means = df.groupby("episode")["reward"].mean()
        completed_pct = df["completed"].astype(bool).mean() * 100
        avg_apples = df["apples_collected"].mean()

        summary.append({
            "model": name,
            "mean_reward": ep_means.mean(),
            "completion_rate": completed_pct,
            "avg_apples": avg_apples
        })

    sum_df = pd.DataFrame(summary)

    # Display name for titles
    mode_title = mode.capitalize()

    # 1. Mean Reward Comparison
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(sum_df["model"], sum_df["mean_reward"], color="#4363D8")
    ax.set(
        xlabel="Model",
        ylabel=f"Mean Greedy {mode_title} Reward",
        title=f"Greedy {mode_title}: Mean Reward Comparison (10 Episodes)"
    )
    ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig.savefig(graph_dir / f"{mode}_rewards_comparison.png", dpi=250)
    plt.close(fig)

    # 2. Level Completion Rate Comparison
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(sum_df["model"], sum_df["completion_rate"], color="#3CB44B")
    ax.set(
        xlabel="Model",
        ylabel="Completion Rate (%)",
        title=f"Greedy {mode_title}: Level Completion Rate (%)"
    )
    ax.set_ylim(0, 105)
    ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig.savefig(graph_dir / f"{mode}_completion_rates.png", dpi=250)
    plt.close(fig)

    # 3. Per-Episode Reward Trajectory
    fig, ax = plt.subplots(figsize=(10, 6))

    for name, df in model_test_dfs.items():
        ep_means = df.groupby("episode")["reward"].mean()
        ax.plot(
            ep_means.index,
            ep_means.values,
            marker="o",
            linewidth=2,
            label=name
        )

    ax.set(
        xlabel="Test Episode",
        ylabel="Mean Episode Reward",
        title=f"Greedy {mode_title}: Reward per Episode Trajectory"
    )
    ax.grid(alpha=0.35)
    ax.legend()
    fig.tight_layout()
    fig.savefig(graph_dir / f"{mode}_episode_progression.png", dpi=250)
    plt.close(fig)

    # 4. Average Apples Collected
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(
        sum_df["model"],
        sum_df["avg_apples"],
        color="#FFE119",
        edgecolor="black"
    )
    ax.set(
        xlabel="Model",
        ylabel="Average Apples Collected",
        title=f"Greedy {mode_title}: Average Apples Collected"
    )
    ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig.savefig(graph_dir / f"{mode}_apple_collection.png", dpi=250)
    plt.close(fig)

    print(f"Saved 4 {mode} result graphs in {graph_dir}/")

In [ ]:
generate_plots(experiments, "testing")

### Cell 17 — Model Fitting Diagnostic: Overfitting vs. Underfitting Analysis
Compares the asymptotic **Training Performance** (late 10% training episodes) against **Validation Performance** (greedy validation episodes) for each model to rigorously diagnose:
- **Underfitted Models:** Poor performance on both training and validation sets (low reward, low completion rate, high mortality). Typically caused by insufficient network capacity, low discount factor $\gamma$, severe under-exploration, or lethal exploration noise.
- **Overfitted Models:** High performance during training on the fixed level seed, but suffering a substantial **Generalization Drop** ($\Delta \ge 15\%$) on unseen validation seeds.
- **Well-Fitted / Generalizing Models:** High training performance that successfully transfers to validation seeds with minimal or positive generalization gap.
- **Diverged / Degenerate Models:** Unstable runs where Q-values or losses experienced mathematical explosion, yielding degraded policies.


In [22]:
def analyze_overfitting_underfitting(experiment_list, train_dir="results/data", val_dir="results/data/testing"):
    """Compares training vs. testing metrics to identify underfitting and overfitting across all models."""
    train_path = Path(train_dir)
    val_path = Path(val_dir)
    
    records = []
    for overrides in experiment_list:
        cfg = build_config(overrides)
        name = cfg["name"]
        tr_file = train_path / f"{name}.csv"
        val_file = val_path / f"{name}_test_results.csv"
        
        if not tr_file.exists():
            continue
            
        tr_df = pd.read_csv(tr_file)
        max_ep = tr_df["episode"].max()
        tr_late = tr_df[tr_df["episode"] >= max_ep * 0.9]
        
        tr_rew = tr_late["reward"].mean()
        tr_comp = tr_late["completed"].astype(bool).mean() * 100
        tr_apples = tr_late["apples_collected"].mean()
        tr_loss_max = tr_df["loss"].max()
        tr_q_max = tr_df["q_max"].max()
        
        if val_file.exists():
            val_df = pd.read_csv(val_file)
            val_rew = val_df["reward"].mean()
            val_comp = val_df["completed"].astype(bool).mean() * 100
            val_apples = val_df["apples_collected"].mean()
        else:
            val_rew, val_comp, val_apples = np.nan, np.nan, np.nan
            
        comp_gap = val_comp - tr_comp if not np.isnan(val_comp) else np.nan
        rew_gap = val_rew - tr_rew if not np.isnan(val_rew) else np.nan
        
        # Diagnostic Classification Logic
        if tr_loss_max > 2.0 or tr_q_max > 500.0:
            status = "Diverged / Degenerate"
            diagnosis = "Severe Q-value / loss explosion during training; numerical instability."
        elif tr_comp < 30.0 and (np.isnan(val_comp) or val_comp < 30.0):
            status = "Underfitted"
            diagnosis = "Low training & validation completion; insufficient capacity, short horizon, or high noise."
        elif comp_gap is not None and comp_gap < -15.0:
            status = "Overfitted (Generalization Gap)"
            diagnosis = f"High training completion ({tr_comp:.1f}%), but validation dropped by {abs(comp_gap):.1f}% on unseen seeds."
        elif val_comp is not None and val_comp >= 45.0:
            status = "Well-Fitted / Robust"
            diagnosis = f"Strong validation performance ({val_comp:.1f}% completion); policy generalizes effectively."
        else:
            status = "Moderately Fitted"
            diagnosis = "Moderate competency with expected generalization variance."
            
        records.append({
            "Model": name,
            "Train Comp (%)": round(tr_comp, 1),
            "Val Comp (%)": round(val_comp, 1) if not np.isnan(val_comp) else "N/A",
            "Comp Gap (%)": round(comp_gap, 1) if not np.isnan(comp_gap) else "N/A",
            "Train Rew": round(tr_rew, 2),
            "Val Rew": round(val_rew, 2) if not np.isnan(val_rew) else "N/A",
            "Diagnosis": status,
            "Details": diagnosis
        })
        
    diag_df = pd.DataFrame(records)
    print("\n" + "=" * 100)
    print("MODEL FITTING & GENERALIZATION DIAGNOSTIC REPORT (TRAINING VS. VALIDATION)")
    print("=" * 100)
    print(diag_df.to_string(index=False))
    print("=" * 100 + "\n")
    
    # Save summary table
    table_dir = Path("results/tables")
    table_dir.mkdir(parents=True, exist_ok=True)
    diag_df.to_csv(table_dir / "model_fitting_analysis.csv", index=False)
    print("Saved diagnostic table to results/tables/model_fitting_analysis.csv")
    return diag_df

# Run fitting analysis
fitting_summary = analyze_overfitting_underfitting(experiments)



MODEL FITTING & GENERALIZATION DIAGNOSTIC REPORT (TRAINING VS. VALIDATION)
          Model  Train Comp (%)  Val Comp (%)  Comp Gap (%)  Train Rew  Val Rew                       Diagnosis                                                                                    Details
baseline_seed42            30.4          67.5          37.1      31.14    71.61            Well-Fitted / Robust          Strong validation performance (67.5% completion); policy generalizes effectively.
  eager_learner             9.4           2.5          -6.9      14.72     3.78           Diverged / Degenerate                    Severe Q-value / loss explosion during training; numerical instability.
       megamind            45.8          57.5          11.7      48.20    62.74            Well-Fitted / Robust          Strong validation performance (57.5% completion); policy generalizes effectively.
           nerd            67.0          42.5         -24.5      65.27    49.31 Overfitted (Generalization Gap) 